# EXP21E-v8 — Mechanism Audit and Statistical Analysis

Post-hoc analysis of immutable V5–V7 artifacts. This notebook does not train, tune, or update any checkpoint. V5 source modules are embedded only to reproduce causal inference and 128-d embeddings from the frozen checkpoints.


In [ ]:
%%writefile data_preprocessing.py
import pandas as pd
import numpy as np
from datetime import timedelta
from sklearn.preprocessing import StandardScaler


def load_and_split_raw_data(
    file_path: str,
    start_year: int = 2016,
    nrows: int = None,
    reference_date: str = "2018-01-01",
    train_days: int = 300,
    val_days: int = 30,
    gap_days: int = 8,
    test_days: int = 60,
) -> tuple:
    """
    Memuat dataset mentah IBM Credit Card Transactions, memfilter era modern (Year >= start_year),
    melakukan parsing datetime kronologis, dan memotongnya ke dalam 4 partisi temporal SHINE FRAUD II
    SEBELUM proses rekayasa fitur atau penskalaan (Strict Out-of-Sample Protocol).
    """
    print(
        f"Loading data from {file_path} (Filter: Year >= {start_year}, nrows={nrows})..."
    )

    # 1. Pemuatan chunked efisien jika start_year ditentukan
    if start_year is not None:
        chunks = []
        collected_rows = 0
        chunksize = 1_000_000

        for chunk in pd.read_csv(file_path, chunksize=chunksize):
            filtered_chunk = chunk[chunk["Year"] >= start_year]
            if len(filtered_chunk) > 0:
                if nrows is not None:
                    remaining = nrows - collected_rows
                    if remaining <= 0:
                        break
                    filtered_chunk = filtered_chunk.iloc[:remaining]
                chunks.append(filtered_chunk)
                collected_rows += len(filtered_chunk)
                if nrows is not None and collected_rows >= nrows:
                    break

        if not chunks:
            raise ValueError(
                f"Tidak ada baris yang ditemukan dengan Year >= {start_year} pada file {file_path}"
            )

        df = pd.concat(chunks, ignore_index=True)
    else:
        df = pd.read_csv(file_path, nrows=nrows)

    print(f"Memproses {len(df)} baris data mentah...")
    # Deterministic identifier within the filtered source. It survives sorting and
    # is stored with predictions so paired sampler outputs can be aligned safely.
    df["transaction_id"] = np.arange(len(df), dtype=np.int64)
    print("Parsing timestamps...")
    df["Time"] = df["Time"].fillna("00:00")
    df["datetime"] = pd.to_datetime(
        df["Year"].astype(str)
        + "-"
        + df["Month"].astype(str).str.zfill(2)
        + "-"
        + df["Day"].astype(str).str.zfill(2)
        + " "
        + df["Time"]
    )

    print("Sorting chronologically...")
    df = df.sort_values("datetime").reset_index(drop=True)

    print("Cleaning Amount & Target Label...")
    df["Amount"] = (
        df["Amount"]
        .astype(str)
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
        .astype(float)
    )
    df["Is Fraud?"] = df["Is Fraud?"].map({"Yes": 1, "No": 0})

    # 2. Pembagian Jendela Waktu Temporal (SHINE FRAUD II)
    print("\nMelakukan pembagian temporal window strictly chronologic...")
    t0 = pd.to_datetime(reference_date)
    t_val_start = t0 + timedelta(days=train_days)
    t_gap_start = t_val_start + timedelta(days=val_days)
    t_test_start = t_gap_start + timedelta(days=gap_days)
    t_test_end = t_test_start + timedelta(days=test_days)

    train_df = (
        df[(df["datetime"] >= t0) & (df["datetime"] < t_val_start)]
        .copy()
        .reset_index(drop=True)
    )
    val_df = (
        df[(df["datetime"] >= t_val_start) & (df["datetime"] < t_gap_start)]
        .copy()
        .reset_index(drop=True)
    )
    gap_df = (
        df[(df["datetime"] >= t_gap_start) & (df["datetime"] < t_test_start)]
        .copy()
        .reset_index(drop=True)
    )
    test_df = (
        df[(df["datetime"] >= t_test_start) & (df["datetime"] < t_test_end)]
        .copy()
        .reset_index(drop=True)
    )

    boundaries = {
        "train": (train_df, t0, t_val_start),
        "validation": (val_df, t_val_start, t_gap_start),
        "embargo_gap": (gap_df, t_gap_start, t_test_start),
        "test": (test_df, t_test_start, t_test_end),
    }
    for split_name, (split_df, lower, upper) in boundaries.items():
        if len(split_df) and not (
            (split_df["datetime"] >= lower).all()
            and (split_df["datetime"] < upper).all()
        ):
            raise AssertionError(f"Temporal boundary violation in split: {split_name}")

    def compute_split_stats(
        name: str,
        split_df: pd.DataFrame,
        start_dt,
        end_dt,
        include_label_stats: bool = True,
    ):
        n_tx = len(split_df)
        if n_tx == 0:
            return {
                "name": name,
                "start_date": str(start_dt),
                "end_date": str(end_dt),
                "num_transactions": 0,
                "num_fraud": 0 if include_label_stats else None,
                "fraud_rate_pct": 0.0 if include_label_stats else None,
                "unique_users": 0,
                "unique_merchants": 0,
                "total_nodes": 0,
            }
        n_fraud = int(split_df["Is Fraud?"].sum()) if include_label_stats else None
        n_users = int(split_df["User"].nunique())
        n_merchants = int(split_df["Merchant Name"].nunique())
        return {
            "name": name,
            "start_date": str(start_dt),
            "end_date": str(end_dt),
            "num_transactions": n_tx,
            "num_fraud": n_fraud,
            "fraud_rate_pct": round((n_fraud / n_tx) * 100, 4)
            if include_label_stats
            else None,
            "unique_users": n_users,
            "unique_merchants": n_merchants,
            "total_nodes": n_tx + n_users + n_merchants,
        }

    total_window_df = pd.concat([train_df, val_df, gap_df, test_df], ignore_index=True)

    stats_dict = {
        "protocol": "SHINE FRAUD II Rolling Window Alignment (EXP21E Pure End-to-End)",
        "reference_date_T0": str(t0),
        "durations_days": {
            "train": train_days,
            "validation": val_days,
            "embargo_gap": gap_days,
            "test": test_days,
            "total_horizon": train_days + val_days + gap_days + test_days,
        },
        "splits": {
            "train": compute_split_stats("Train", train_df, t0, t_val_start),
            "validation": compute_split_stats(
                "Validation", val_df, t_val_start, t_gap_start
            ),
            "embargo_gap": compute_split_stats(
                "Embargo Gap", gap_df, t_gap_start, t_test_start
            ),
            # Test-label summaries are intentionally deferred until the frozen
            # checkpoint and validation threshold are evaluated in main.py.
            "test": compute_split_stats(
                "Test", test_df, t_test_start, t_test_end, include_label_stats=False
            ),
            "total_window": compute_split_stats(
                "Total Window",
                total_window_df,
                t0,
                t_test_end,
                include_label_stats=False,
            ),
        },
    }

    return train_df, val_df, test_df, gap_df, stats_dict


def apply_strict_anti_leakage_preprocessing(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    test_df: pd.DataFrame,
    gap_df: pd.DataFrame = None,
    feature_mode: str = "clean",
    use_cyclical_time: bool = True,
    use_amount_log: bool = True,
) -> tuple:
    """
    Menerapkan Strict Anti-Leakage Preprocessing Protocol (EXP21E):
    1. StandardScaler Amount HANYA di-fit pada train_df, lalu mentransformasikan val_df, test_df, gap_df.
    2. Encoding kategorikal HANYA mempelajari kamus nilai unik dari train_df. Nilai baru pada val/test dipetakan ke 0 (<oov>).
    3. Mengeliminasi Zip dan Merchant City (mode: clean) untuk mencegah pohon keputusan menghafal ID kuasi.
    4. Menambahkan fitur waktu siklik (sin/cos jam & hari) yang bebas memorization.
    5. Menambahkan amount_log (log1p Amount) untuk menstabilkan distribusi numerik.
    """
    print(
        f"\n[STRICT ANTI-LEAKAGE] Menerapkan preprocessing (Mode: {feature_mode}, Cyclical Time: {use_cyclical_time}, Amount Log: {use_amount_log})..."
    )

    # Simpan nilai Amount mentah positif untuk log amount
    for df_split in [train_df, val_df, test_df] + (
        [gap_df] if gap_df is not None and len(gap_df) > 0 else []
    ):
        if use_amount_log:
            df_split["amount_log"] = np.log1p(
                np.maximum(0.0, df_split["Amount"].values)
            ).astype(np.float32)

    # Tentukan atribut kategorikal yang diizinkan
    if feature_mode == "clean":
        cat_columns = ["Use Chip", "Merchant State", "MCC", "Errors?"]
        print("  • Fitur High-Cardinality ('Zip', 'Merchant City') DIHAPUS dari input!")
    elif feature_mode == "clean_basic":
        cat_columns = ["Use Chip", "MCC", "Errors?"]
        print(f"  • Menggunakan fitur kategorikal minimal: {cat_columns}")
    else:
        raise ValueError(
            f"Feature mode '{feature_mode}' tidak dikenal. Pilih 'clean' atau 'clean_basic'."
        )

    # A. Standarisasi Fitur Numerik (Amount) — FIT HANYA DI TRAIN
    scaler_amount = StandardScaler()
    train_df["Amount"] = scaler_amount.fit_transform(train_df[["Amount"]])
    val_df["Amount"] = scaler_amount.transform(val_df[["Amount"]])
    test_df["Amount"] = scaler_amount.transform(test_df[["Amount"]])
    if gap_df is not None and len(gap_df) > 0:
        gap_df["Amount"] = scaler_amount.transform(gap_df[["Amount"]])

    print(
        f"  • Amount Scaler fit pada Train: mean={scaler_amount.mean_[0]:.4f}, std={scaler_amount.scale_[0]:.4f}"
    )

    if use_amount_log:
        scaler_log = StandardScaler()
        train_df["amount_log"] = scaler_log.fit_transform(train_df[["amount_log"]])
        val_df["amount_log"] = scaler_log.transform(val_df[["amount_log"]])
        test_df["amount_log"] = scaler_log.transform(test_df[["amount_log"]])
        if gap_df is not None and len(gap_df) > 0:
            gap_df["amount_log"] = scaler_log.transform(gap_df[["amount_log"]])
        print(
            f"  • Amount Log Scaler fit pada Train: mean={scaler_log.mean_[0]:.4f}, std={scaler_log.scale_[0]:.4f}"
        )

    # B. Robust Categorical Encoding — DICTIONARY FIT HANYA DI TRAIN
    categorical_encoders = {}
    for col in cat_columns:
        train_df[col] = train_df[col].fillna("<missing>").astype(str)
        val_df[col] = val_df[col].fillna("<missing>").astype(str)
        test_df[col] = test_df[col].fillna("<missing>").astype(str)
        if gap_df is not None and len(gap_df) > 0:
            gap_df[col] = gap_df[col].fillna("<missing>").astype(str)

        # Pelajari mapping unik dari TRAIN saja
        unique_train_vals = sorted(train_df[col].unique().tolist())
        # Index 0 dicadangkan untuk OOV (<unknown>), kategori train mulai dari index 1
        cat_to_id = {val: idx + 1 for idx, val in enumerate(unique_train_vals)}
        categorical_encoders[col] = cat_to_id

        # Transform dengan fallback OOV = 0
        train_df[col] = train_df[col].map(cat_to_id).fillna(0).astype(int)
        val_df[col] = val_df[col].map(cat_to_id).fillna(0).astype(int)
        test_df[col] = test_df[col].map(cat_to_id).fillna(0).astype(int)
        if gap_df is not None and len(gap_df) > 0:
            gap_df[col] = gap_df[col].map(cat_to_id).fillna(0).astype(int)

        num_oov_val = int((val_df[col] == 0).sum())
        num_oov_test = int((test_df[col] == 0).sum())
        print(
            f"  • Kolom '{col}': {len(cat_to_id)} kategori train dipelajari (OOV di Val: {num_oov_val}, di Test: {num_oov_test})"
        )

    # C. Fitur Temporal Siklik (Cyclical Time Features)
    feature_cols = ["Amount"]
    if use_amount_log:
        feature_cols.append("amount_log")
    feature_cols = feature_cols + cat_columns

    if use_cyclical_time:
        all_splits = [train_df, val_df, test_df]
        if gap_df is not None and len(gap_df) > 0:
            all_splits.append(gap_df)

        for split_df in all_splits:
            hour_float = (
                split_df["datetime"].dt.hour + split_df["datetime"].dt.minute / 60.0
            )
            split_df["hour_sin"] = np.sin(2 * np.pi * hour_float / 24.0).astype(
                np.float32
            )
            split_df["hour_cos"] = np.cos(2 * np.pi * hour_float / 24.0).astype(
                np.float32
            )

            dow = split_df["datetime"].dt.dayofweek
            split_df["dow_sin"] = np.sin(2 * np.pi * dow / 7.0).astype(np.float32)
            split_df["dow_cos"] = np.cos(2 * np.pi * dow / 7.0).astype(np.float32)

        time_cols = ["hour_sin", "hour_cos", "dow_sin", "dow_cos"]
        feature_cols = feature_cols + time_cols
        print(f"  • Ditambahkan 4 fitur waktu siklik: {time_cols}")

    print(
        f"Total fitur transaksi bersih (feature_cols): {len(feature_cols)} fitur -> {feature_cols}"
    )

    artifacts = {
        "scaler_amount": scaler_amount,
        "categorical_encoders": categorical_encoders,
        "feature_cols": feature_cols,
        "feature_mode": feature_mode,
    }

    return train_df, val_df, test_df, gap_df, feature_cols, artifacts


In [ ]:
%%writefile graph_builder.py
import torch
import numpy as np
import pandas as pd
from torch_geometric.data import HeteroData
from sklearn.preprocessing import StandardScaler


def graph_level_undersample(
    df: pd.DataFrame, ratio: float = 1.0, seed: int = 42
) -> pd.DataFrame:
    """
    Melakukan Graph-Level Undersampling jika ratio < 1.0.
    Jika ratio == 1.0 (default di exp21d/e & SHINE), seluruh data dipertahankan
    tanpa merusak topologi asli jaringan.
    """
    if ratio >= 1.0:
        return df

    print(f"Menerapkan graph-level undersampling dengan rasio {ratio}...")
    labels = df["Is Fraud?"].values
    fraud_idx = np.flatnonzero(labels == 1)
    normal_idx = np.flatnonzero(labels == 0)

    if len(fraud_idx) == 0 or len(normal_idx) == 0:
        return df

    desired_normal = int(np.floor(len(normal_idx) * ratio))
    if desired_normal > len(normal_idx):
        desired_normal = len(normal_idx)

    rng = np.random.default_rng(seed)
    chosen_normal = rng.choice(normal_idx, size=desired_normal, replace=False)

    selected_idx = np.concatenate([chosen_normal, fraud_idx])
    selected_idx.sort()

    return df.iloc[selected_idx].reset_index(drop=True)


def build_hetero_graph(
    df: pd.DataFrame, feature_cols: list, scalers_dict: dict = None
) -> tuple:
    """
    Membangun graf heterogen tripartit PyTorch Geometric (HeteroData).
    Nodes:
      - transaction: Fitur transaksi bersih (feature_cols) & label target
      - user: Agregat rata-rata Amount & jumlah transaksi
      - merchant: Agregat rata-rata Amount & jumlah transaksi
    Edges:
      - (user, buys, transaction) & (transaction, rev_buys, user)
      - (merchant, sells, transaction) & (transaction, rev_sells, merchant)

    Anti-Leakage Guard:
    Jika scalers_dict is None, scaler untuk User dan Merchant di-fit pada data saat ini (hanya untuk Train).
    Jika scalers_dict disediakan, scaler dari Train digunakan untuk me-transform User dan Merchant
    pada Validation dan Test tanpa membocorkan statistik masa depan.
    """
    print(f"Membangun HeteroData graf ({len(df)} transaksi)...")
    data = HeteroData()

    # 1. Pemetaan ID User dan Merchant ke integer kontigu
    user_mapping = {uid: i for i, uid in enumerate(df["User"].unique())}
    merchant_mapping = {mid: i for i, mid in enumerate(df["Merchant Name"].unique())}

    num_users = len(user_mapping)
    num_merchants = len(merchant_mapping)
    num_transactions = len(df)

    # 2. Agregasi fitur User dan Merchant
    user_agg = (
        df.groupby("User")
        .agg(user_avg_amount=("Amount", "mean"), user_tx_count=("Amount", "count"))
        .reset_index()
    )

    merchant_agg = (
        df.groupby("Merchant Name")
        .agg(
            merchant_avg_amount=("Amount", "mean"),
            merchant_tx_count=("Amount", "count"),
        )
        .reset_index()
    )

    # Pastikan urutan sesuai dengan mapping
    user_agg["mapped_id"] = user_agg["User"].map(user_mapping)
    user_agg = user_agg.sort_values("mapped_id").reset_index(drop=True)

    merchant_agg["mapped_id"] = merchant_agg["Merchant Name"].map(merchant_mapping)
    merchant_agg = merchant_agg.sort_values("mapped_id").reset_index(drop=True)

    # Penskalaan fitur User & Merchant (Strict Anti-Leakage)
    if scalers_dict is None:
        scaler_user = StandardScaler()
        user_feats_scaled = scaler_user.fit_transform(
            user_agg[["user_avg_amount", "user_tx_count"]]
        )

        scaler_merchant = StandardScaler()
        merchant_feats_scaled = scaler_merchant.fit_transform(
            merchant_agg[["merchant_avg_amount", "merchant_tx_count"]]
        )

        out_scalers = {"scaler_user": scaler_user, "scaler_merchant": scaler_merchant}
    else:
        scaler_user = scalers_dict["scaler_user"]
        scaler_merchant = scalers_dict["scaler_merchant"]
        user_feats_scaled = scaler_user.transform(
            user_agg[["user_avg_amount", "user_tx_count"]]
        )
        merchant_feats_scaled = scaler_merchant.transform(
            merchant_agg[["merchant_avg_amount", "merchant_tx_count"]]
        )
        out_scalers = scalers_dict

    user_features = torch.tensor(user_feats_scaled, dtype=torch.float)
    merchant_features = torch.tensor(merchant_feats_scaled, dtype=torch.float)

    # 3. Fitur Transaksi & Label
    tx_features = torch.tensor(df[feature_cols].values, dtype=torch.float)
    tx_labels = torch.tensor(df["Is Fraud?"].values, dtype=torch.float)

    data["transaction"].x = tx_features
    data["transaction"].y = tx_labels
    data["transaction"].num_nodes = num_transactions
    data["transaction"].node_id = torch.arange(num_transactions, dtype=torch.long)

    data["user"].x = user_features
    data["user"].num_nodes = num_users
    data["user"].node_id = torch.arange(num_users, dtype=torch.long)

    data["merchant"].x = merchant_features
    data["merchant"].num_nodes = num_merchants
    data["merchant"].node_id = torch.arange(num_merchants, dtype=torch.long)

    # 4. Membangun Edge Index Bipartit
    tx_indices = np.arange(num_transactions)
    user_src_indices = df["User"].map(user_mapping).values
    merchant_src_indices = df["Merchant Name"].map(merchant_mapping).values

    edge_user_to_tx = torch.tensor(
        np.array([user_src_indices, tx_indices]), dtype=torch.long
    )
    edge_tx_to_user = torch.tensor(
        np.array([tx_indices, user_src_indices]), dtype=torch.long
    )

    edge_merchant_to_tx = torch.tensor(
        np.array([merchant_src_indices, tx_indices]), dtype=torch.long
    )
    edge_tx_to_merchant = torch.tensor(
        np.array([tx_indices, merchant_src_indices]), dtype=torch.long
    )

    data["user", "buys", "transaction"].edge_index = edge_user_to_tx
    data["transaction", "rev_buys", "user"].edge_index = edge_tx_to_user

    data["merchant", "sells", "transaction"].edge_index = edge_merchant_to_tx
    data["transaction", "rev_sells", "merchant"].edge_index = edge_tx_to_merchant

    # Stable edge IDs are retained by sampled mini-batches for sampler auditing.
    for edge_type in data.edge_types:
        data[edge_type].edge_id = torch.arange(
            data[edge_type].edge_index.size(1), dtype=torch.long
        )

    print(
        f"HeteroData selesai: {num_transactions} tx, {num_users} users, {num_merchants} merchants."
    )
    return data, out_scalers


In [ ]:
%%writefile causal_graph.py
from __future__ import annotations

import hashlib
from dataclasses import dataclass
from typing import Any, Iterator

import numpy as np
import pandas as pd
import torch
from torch_geometric.data import HeteroData


USER_TO_TX = ("user", "buys", "transaction")
TX_TO_USER = ("transaction", "rev_buys", "user")
MERCHANT_TO_TX = ("merchant", "sells", "transaction")
TX_TO_MERCHANT = ("transaction", "rev_sells", "merchant")


@dataclass
class CausalDay:
    day: str
    graph: HeteroData
    root_nodes: torch.Tensor
    target_frame: pd.DataFrame
    audit: dict[str, Any]


class CausalHistoryState:
    """Incremental label-free history used by the daily-batch evaluator."""

    def __init__(self, initial_history: pd.DataFrame):
        self.frame = (
            initial_history.sort_values("datetime").reset_index(drop=True).copy()
        )
        self.user_rows: dict[Any, list[int]] = {}
        self.merchant_rows: dict[Any, list[int]] = {}
        self.user_amount_sum: dict[Any, float] = {}
        self.merchant_amount_sum: dict[Any, float] = {}
        self.user_count: dict[Any, int] = {}
        self.merchant_count: dict[Any, int] = {}
        self._index_rows(self.frame, offset=0)

    def _index_rows(self, frame: pd.DataFrame, offset: int) -> None:
        # Deliberately do not read `Is Fraud?`: graph state and scores are label-free.
        for local_index, (user, merchant, amount) in enumerate(
            frame[["User", "Merchant Name", "Amount"]].itertuples(
                index=False, name=None
            )
        ):
            row_index = offset + local_index
            self.user_rows.setdefault(user, []).append(row_index)
            self.merchant_rows.setdefault(merchant, []).append(row_index)
            self.user_amount_sum[user] = self.user_amount_sum.get(user, 0.0) + float(
                amount
            )
            self.merchant_amount_sum[merchant] = self.merchant_amount_sum.get(
                merchant, 0.0
            ) + float(amount)
            self.user_count[user] = self.user_count.get(user, 0) + 1
            self.merchant_count[merchant] = self.merchant_count.get(merchant, 0) + 1

    def append(self, completed_day: pd.DataFrame) -> None:
        if completed_day.empty:
            return
        completed = completed_day.sort_values("datetime").reset_index(drop=True).copy()
        if (
            len(self.frame)
            and completed["datetime"].min() <= self.frame["datetime"].max()
        ):
            raise ValueError(
                "Causal history must be appended in strictly increasing time order"
            )
        offset = len(self.frame)
        self._index_rows(completed, offset=offset)
        self.frame = pd.concat([self.frame, completed], ignore_index=True)

    def relevant_context(self, target_frame: pd.DataFrame) -> pd.DataFrame:
        row_ids: set[int] = set()
        for user in target_frame["User"].unique():
            row_ids.update(self.user_rows.get(user, ()))
        for merchant in target_frame["Merchant Name"].unique():
            row_ids.update(self.merchant_rows.get(merchant, ()))
        if not row_ids:
            return self.frame.iloc[0:0].copy()
        return self.frame.iloc[sorted(row_ids)].copy().reset_index(drop=True)

    def entity_features(self, values: list[Any], entity_type: str) -> np.ndarray:
        if entity_type == "user":
            sums, counts = self.user_amount_sum, self.user_count
        elif entity_type == "merchant":
            sums, counts = self.merchant_amount_sum, self.merchant_count
        else:
            raise ValueError(f"Unsupported entity type: {entity_type}")
        output = np.zeros((len(values), 2), dtype=np.float64)
        for index, value in enumerate(values):
            count = counts.get(value, 0)
            output[index, 0] = sums.get(value, 0.0) / count if count else 0.0
            output[index, 1] = count
        return output

    def degrees_for(self, frame: pd.DataFrame) -> tuple[np.ndarray, np.ndarray]:
        user_degree = (
            frame["User"].map(self.user_count).fillna(0).to_numpy(dtype=np.float32)
        )
        merchant_degree = (
            frame["Merchant Name"]
            .map(self.merchant_count)
            .fillna(0)
            .to_numpy(dtype=np.float32)
        )
        return user_degree, merchant_degree


def _edge_tensor(source: np.ndarray, destination: np.ndarray) -> torch.Tensor:
    if len(source) != len(destination):
        raise ValueError("Edge source and destination arrays must have equal length")
    if len(source) == 0:
        return torch.empty((2, 0), dtype=torch.long)
    return torch.tensor(np.stack([source, destination]), dtype=torch.long)


def _timestamp_ns(frame: pd.DataFrame) -> np.ndarray:
    return frame["datetime"].astype("int64").to_numpy(dtype=np.int64)


def build_causal_day_graph(
    history: CausalHistoryState,
    target_frame: pd.DataFrame,
    feature_cols: list[str],
    entity_scalers: dict[str, Any],
) -> CausalDay:
    """Build the exact two-hop graph needed to score one frozen calendar day."""
    if target_frame.empty:
        raise ValueError("A causal day cannot be empty")
    target = (
        target_frame.sort_values(["datetime", "transaction_id"])
        .reset_index(drop=True)
        .copy()
    )
    normalized_days = target["datetime"].dt.normalize().unique()
    if len(normalized_days) != 1:
        raise ValueError("target_frame must contain exactly one calendar day")

    cutoff = pd.Timestamp(normalized_days[0])
    day_end = cutoff + pd.Timedelta(days=1)
    if len(history.frame) and history.frame["datetime"].max() >= cutoff:
        raise AssertionError(
            "History contains information at or after the target-day cutoff"
        )
    if not ((target["datetime"] >= cutoff) & (target["datetime"] < day_end)).all():
        raise AssertionError("Target rows fall outside the frozen daily batch")

    context = history.relevant_context(target)
    num_context = len(context)
    combined = pd.concat([context, target], ignore_index=True)
    root_nodes = torch.arange(num_context, len(combined), dtype=torch.long)

    users = combined["User"].drop_duplicates().tolist()
    merchants = combined["Merchant Name"].drop_duplicates().tolist()
    user_mapping = {value: index for index, value in enumerate(users)}
    merchant_mapping = {value: index for index, value in enumerate(merchants)}
    user_ids = combined["User"].map(user_mapping).to_numpy(dtype=np.int64)
    merchant_ids = (
        combined["Merchant Name"].map(merchant_mapping).to_numpy(dtype=np.int64)
    )
    transaction_ids = np.arange(len(combined), dtype=np.int64)
    context_transaction_ids = transaction_ids[:num_context]

    user_features = history.entity_features(users, "user")
    merchant_features = history.entity_features(merchants, "merchant")
    user_feature_frame = pd.DataFrame(
        user_features, columns=["user_avg_amount", "user_tx_count"]
    )
    merchant_feature_frame = pd.DataFrame(
        merchant_features, columns=["merchant_avg_amount", "merchant_tx_count"]
    )
    user_features = entity_scalers["scaler_user"].transform(user_feature_frame)
    merchant_features = entity_scalers["scaler_merchant"].transform(
        merchant_feature_frame
    )
    historical_user_degree, historical_merchant_degree = history.degrees_for(combined)

    data = HeteroData()
    data["transaction"].x = torch.tensor(
        combined[feature_cols].to_numpy(), dtype=torch.float32
    )
    labels = np.zeros(len(combined), dtype=np.float32)
    labels[num_context:] = target["Is Fraud?"].to_numpy(dtype=np.float32)
    data["transaction"].y = torch.tensor(labels, dtype=torch.float32)
    data["transaction"].num_nodes = len(combined)
    data["transaction"].node_id = torch.arange(len(combined), dtype=torch.long)
    data["transaction"].transaction_id = torch.tensor(
        combined["transaction_id"].to_numpy(dtype=np.int64), dtype=torch.long
    )
    data["transaction"].timestamp_ns = torch.tensor(
        _timestamp_ns(combined), dtype=torch.long
    )
    data["transaction"].is_target = torch.zeros(len(combined), dtype=torch.bool)
    data["transaction"].is_target[root_nodes] = True
    # These global historical degrees keep candidate scoring exact even though the
    # materialized graph contains only the two-hop context relevant to today's roots.
    data["transaction"].historical_user_degree = torch.tensor(
        historical_user_degree, dtype=torch.float32
    )
    data["transaction"].historical_merchant_degree = torch.tensor(
        historical_merchant_degree, dtype=torch.float32
    )

    data["user"].x = torch.tensor(user_features, dtype=torch.float32)
    data["user"].num_nodes = len(users)
    data["user"].node_id = torch.arange(len(users), dtype=torch.long)
    data["merchant"].x = torch.tensor(merchant_features, dtype=torch.float32)
    data["merchant"].num_nodes = len(merchants)
    data["merchant"].node_id = torch.arange(len(merchants), dtype=torch.long)

    data[USER_TO_TX].edge_index = _edge_tensor(user_ids, transaction_ids)
    data[TX_TO_USER].edge_index = _edge_tensor(
        context_transaction_ids, user_ids[:num_context]
    )
    data[MERCHANT_TO_TX].edge_index = _edge_tensor(merchant_ids, transaction_ids)
    data[TX_TO_MERCHANT].edge_index = _edge_tensor(
        context_transaction_ids, merchant_ids[:num_context]
    )

    combined_timestamp = _timestamp_ns(combined)
    stable_edge_ids = combined["transaction_id"].to_numpy(dtype=np.int64)
    for edge_type in (USER_TO_TX, MERCHANT_TO_TX):
        data[edge_type].edge_id = torch.tensor(stable_edge_ids, dtype=torch.long)
        data[edge_type].edge_timestamp_ns = torch.tensor(
            combined_timestamp, dtype=torch.long
        )
    for edge_type in (TX_TO_USER, TX_TO_MERCHANT):
        data[edge_type].edge_id = torch.tensor(
            stable_edge_ids[:num_context], dtype=torch.long
        )
        data[edge_type].edge_timestamp_ns = torch.tensor(
            combined_timestamp[:num_context], dtype=torch.long
        )

    cutoff_ns = int(cutoff.value)
    reverse_timestamp = combined_timestamp[:num_context]
    no_future_candidates = bool(
        reverse_timestamp.size == 0 or np.all(reverse_timestamp < cutoff_ns)
    )
    target_digest = hashlib.sha256(
        target["transaction_id"].to_numpy(dtype=np.int64).tobytes()
    ).hexdigest()
    audit = {
        "day": cutoff.date().isoformat(),
        "cutoff_policy": "history_strictly_before_start_of_day",
        "history_rows": len(history.frame),
        "materialized_context_rows": num_context,
        "target_rows": len(target),
        "target_transaction_id_sha256": target_digest,
        "history_max_timestamp": (
            str(history.frame["datetime"].max()) if len(history.frame) else None
        ),
        "target_min_timestamp": str(target["datetime"].min()),
        "target_max_timestamp": str(target["datetime"].max()),
        "candidate_max_timestamp": str(context["datetime"].max())
        if len(context)
        else None,
        "target_reverse_edges": 0,
        "aggregate_source": "label_free_history_before_cutoff",
        "sampling_score_source": "historical_counterpart_degree_before_cutoff",
        "no_future_candidates": no_future_candidates,
        "passed": bool(no_future_candidates),
    }
    if not audit["passed"]:
        raise AssertionError(f"Causal graph audit failed for {audit['day']}")
    return CausalDay(
        day=audit["day"],
        graph=data,
        root_nodes=root_nodes,
        target_frame=target,
        audit=audit,
    )


def prepare_causal_daily_plan(
    initial_history: pd.DataFrame,
    target_frame: pd.DataFrame,
    feature_cols: list[str],
    entity_scalers: dict[str, Any],
    max_days: int | None = None,
) -> tuple[list[CausalDay], CausalHistoryState]:
    state = CausalHistoryState(initial_history)
    plan = list(
        iter_causal_daily_days(
            state=state,
            target_frame=target_frame,
            feature_cols=feature_cols,
            entity_scalers=entity_scalers,
            max_days=max_days,
        )
    )
    return plan, state


def iter_causal_daily_days(
    state: CausalHistoryState,
    target_frame: pd.DataFrame,
    feature_cols: list[str],
    entity_scalers: dict[str, Any],
    max_days: int | None = None,
) -> Iterator[CausalDay]:
    """Yield and release one causal day at a time for memory-bounded test inference."""
    target = target_frame.sort_values(["datetime", "transaction_id"]).reset_index(
        drop=True
    )
    day_values = target["datetime"].dt.normalize()
    unique_days = sorted(day_values.unique())
    if max_days is not None:
        unique_days = unique_days[:max_days]

    for day_value in unique_days:
        day_frame = target.loc[day_values == day_value].copy().reset_index(drop=True)
        causal_day = build_causal_day_graph(
            state, day_frame, feature_cols, entity_scalers
        )
        yield causal_day
        state.append(day_frame)


def plan_audit(plan: list[CausalDay]) -> dict[str, Any]:
    rows = [item.audit for item in plan]
    return {
        "protocol": "daily_batch_frozen_history",
        "days": len(rows),
        "targets": int(sum(row["target_rows"] for row in rows)),
        "all_days_passed": bool(rows and all(row["passed"] for row in rows)),
        "daily": rows,
    }


In [ ]:
%%writefile samplers.py
from __future__ import annotations

from typing import Any

import torch
from torch_geometric.data import HeteroData
from torch_geometric.loader import NeighborLoader


WEIGHT_ATTR = "sampling_weight"
USER_TO_TX = ("user", "buys", "transaction")
TX_TO_USER = ("transaction", "rev_buys", "user")
MERCHANT_TO_TX = ("merchant", "sells", "transaction")
TX_TO_MERCHANT = ("transaction", "rev_sells", "merchant")


def _bincount(index: torch.Tensor, size: int) -> torch.Tensor:
    return torch.bincount(index, minlength=size).to(dtype=torch.float32)


def _minmax_with_floor(values: torch.Tensor, alpha: float) -> torch.Tensor:
    if not 0.0 <= alpha < 1.0:
        raise ValueError(f"alpha must be in [0, 1), received {alpha}")
    values = values.to(dtype=torch.float32)
    minimum = values.min()
    maximum = values.max()
    if torch.isclose(maximum, minimum):
        return torch.ones_like(values)
    scaled = (values - minimum) / (maximum - minimum)
    return alpha + (1.0 - alpha) * scaled


def candidate_score(
    candidate_degree: torch.Tensor, method: str, alpha: float = 0.1
) -> torch.Tensor:
    """Create a stable score for transaction candidates competing at hop two."""
    method = method.lower()
    degree = candidate_degree.to(dtype=torch.float32).clamp_min(1.0)
    if degree.numel() == 0:
        return degree
    if method == "uniform":
        return torch.ones_like(degree)
    if method == "topology":
        raw = 1.0 / torch.log1p(degree)
    elif method == "importance":
        raw = torch.log1p(degree)
    else:
        raise ValueError(f"Unsupported sampling method: {method}")
    return _minmax_with_floor(raw, alpha=alpha)


def _transaction_counterpart_degrees(
    data: HeteroData,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Return each transaction's user degree and merchant degree."""
    required = {USER_TO_TX, TX_TO_USER, MERCHANT_TO_TX, TX_TO_MERCHANT}
    missing = required.difference(data.edge_types)
    if missing:
        raise ValueError(f"Missing required tripartite edge types: {sorted(missing)}")

    transaction_store = data["transaction"]
    if hasattr(transaction_store, "historical_user_degree") and hasattr(
        transaction_store, "historical_merchant_degree"
    ):
        user_degree_per_tx = transaction_store.historical_user_degree.to(
            dtype=torch.float32
        )
        merchant_degree_per_tx = transaction_store.historical_merchant_degree.to(
            dtype=torch.float32
        )
        expected = int(transaction_store.num_nodes)
        if (
            user_degree_per_tx.numel() != expected
            or merchant_degree_per_tx.numel() != expected
        ):
            raise ValueError(
                "Historical counterpart degree shape does not match transaction nodes"
            )
        # New entities legitimately have zero historical degree. candidate_score
        # applies a floor before transforming these values.
        return user_degree_per_tx, merchant_degree_per_tx

    num_transactions = transaction_store.num_nodes
    user_edges = data[USER_TO_TX].edge_index
    merchant_edges = data[MERCHANT_TO_TX].edge_index

    user_degree = _bincount(user_edges[0], data["user"].num_nodes)
    merchant_degree = _bincount(merchant_edges[0], data["merchant"].num_nodes)

    user_for_tx = torch.full((num_transactions,), -1, dtype=torch.long)
    merchant_for_tx = torch.full((num_transactions,), -1, dtype=torch.long)
    user_for_tx[user_edges[1]] = user_edges[0]
    merchant_for_tx[merchant_edges[1]] = merchant_edges[0]

    if (user_for_tx < 0).any() or (merchant_for_tx < 0).any():
        raise ValueError(
            "Every transaction must have exactly one user and merchant counterpart"
        )

    return user_degree[user_for_tx], merchant_degree[merchant_for_tx]


def apply_sampling_weights(
    data: HeteroData,
    method: str,
    alpha: float = 0.1,
    weight_attr: str = WEIGHT_ATTR,
) -> HeteroData:
    """
    Attach weights consumed by PyG NeighborLoader.

    Sampling starts from transaction roots. The first hop has one user and one merchant,
    so no candidate competition occurs there. At hop two, transaction candidates compete:

    * transaction -> user uses the candidate transaction's merchant degree;
    * transaction -> merchant uses the candidate transaction's user degree.
    """
    method = method.lower()
    user_degree_per_tx, merchant_degree_per_tx = _transaction_counterpart_degrees(data)

    for edge_type in data.edge_types:
        num_edges = data[edge_type].edge_index.size(1)
        weights = torch.ones(num_edges, dtype=torch.float32)
        candidate_degree = torch.ones(num_edges, dtype=torch.float32)

        if edge_type == TX_TO_USER:
            tx_ids = data[edge_type].edge_index[0]
            candidate_degree = merchant_degree_per_tx[tx_ids]
            weights = candidate_score(candidate_degree, method=method, alpha=alpha)
        elif edge_type == TX_TO_MERCHANT:
            tx_ids = data[edge_type].edge_index[0]
            candidate_degree = user_degree_per_tx[tx_ids]
            weights = candidate_score(candidate_degree, method=method, alpha=alpha)

        if weights.numel() != num_edges:
            raise RuntimeError(f"Weight shape mismatch for edge type {edge_type}")
        if not torch.isfinite(weights).all() or (weights < 0).any():
            raise ValueError(f"Invalid sampling weights for edge type {edge_type}")

        data[edge_type][weight_attr] = weights.contiguous()
        data[edge_type].candidate_degree = candidate_degree.contiguous()

    validate_local_weight_sums(data, weight_attr=weight_attr)
    return data


def validate_local_weight_sums(
    data: HeteroData, weight_attr: str = WEIGHT_ATTR
) -> None:
    for edge_type in data.edge_types:
        store = data[edge_type]
        weights = store[weight_attr]
        destination = store.edge_index[1]
        sums = torch.zeros(data[edge_type[2]].num_nodes, dtype=torch.float32)
        sums.scatter_add_(0, destination, weights)
        active = torch.bincount(destination, minlength=sums.numel()) > 0
        if active.any() and not (sums[active] > 0).all():
            raise ValueError(f"Non-positive local weight sum for edge type {edge_type}")


def sampling_weight_audit(
    data: HeteroData, weight_attr: str = WEIGHT_ATTR
) -> dict[str, Any]:
    report: dict[str, Any] = {}
    for edge_type in data.edge_types:
        store = data[edge_type]
        weights = store[weight_attr].detach().cpu()
        degrees = store.candidate_degree.detach().cpu()
        edge_name = "__".join(edge_type)
        report[edge_name] = {
            "num_edges": int(weights.numel()),
            "weight_min": float(weights.min()) if weights.numel() else None,
            "weight_max": float(weights.max()) if weights.numel() else None,
            "weight_mean": float(weights.mean()) if weights.numel() else None,
            "weight_std": float(weights.std(unbiased=False))
            if weights.numel()
            else None,
            "unique_weights": int(torch.unique(weights).numel()),
            "non_constant": bool(torch.unique(weights).numel() > 1),
            "candidate_degree_min": float(degrees.min()) if degrees.numel() else None,
            "candidate_degree_max": float(degrees.max()) if degrees.numel() else None,
            "candidate_degree_mean": float(degrees.mean()) if degrees.numel() else None,
        }
    return report


def get_neighbor_loader(
    data: HeteroData,
    method: str = "uniform",
    batch_size: int = 1024,
    shuffle: bool = False,
    num_neighbors: list[int] | None = None,
    seed: int = 42,
    alpha: float = 0.1,
    input_nodes: torch.Tensor | None = None,
    weight_attr: str = WEIGHT_ATTR,
) -> NeighborLoader:
    """Build a reproducible PyG NeighborLoader with active weighted sampling."""
    if num_neighbors is None:
        num_neighbors = [15, 10]
    apply_sampling_weights(data, method=method, alpha=alpha, weight_attr=weight_attr)
    generator = torch.Generator()
    generator.manual_seed(seed)
    if input_nodes is None:
        input_nodes = torch.arange(data["transaction"].num_nodes)

    return NeighborLoader(
        data,
        num_neighbors={edge_type: list(num_neighbors) for edge_type in data.edge_types},
        batch_size=batch_size,
        input_nodes=("transaction", input_nodes),
        shuffle=shuffle,
        num_workers=0,
        weight_attr=weight_attr,
        generator=generator,
        subgraph_type="directional",
    )


In [ ]:
%%writefile model.py
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv, HeteroConv


class HeteroGraphSAGE(nn.Module):
    """
    Arsitektur HeteroGraphSAGE End-to-End sesuai Bab 3.4 Proposal Tugas Akhir:
    - 2 Lapisan HeteroConv (SAGEConv dengan Mean Aggregator)
    - Batch Normalization & ReLU Activation
    - Dropout Regularization
    - Lapisan Klasifikasi Akhir: Linear(hidden_channels -> out_channels=1)
    - Output Logits langsung dipetakan via Sigmoid untuk estimasi probabilitas fraud: p_v = sigmoid(z_v)
    """

    def __init__(
        self,
        hidden_channels: int,
        out_channels: int,
        num_layers: int,
        data_metadata: tuple,
        feature_dims: dict,
        aggr: str = "mean",
        dropout: float = 0.2,
    ):
        super().__init__()

        self.num_layers = num_layers
        self.aggr = aggr
        self.dropout_rate = dropout
        self.dropout = nn.Dropout(dropout) if dropout > 0.0 else nn.Identity()

        # Inisialisasi proyeksi linear untuk tiap tipe node
        self.lin_embeddings = nn.ModuleDict()
        self.param_embeddings = nn.ParameterDict()
        for node_type, dim in feature_dims.items():
            if dim == 0:
                self.param_embeddings[node_type] = nn.Parameter(
                    torch.randn(1, hidden_channels)
                )
            else:
                self.lin_embeddings[node_type] = nn.Linear(dim, hidden_channels)

        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()

        for _ in range(num_layers):
            conv = HeteroConv(
                {
                    edge_type: SAGEConv(
                        in_channels=(-1, -1),  # Lazy initialization
                        out_channels=hidden_channels,
                        aggr=aggr,
                    )
                    for edge_type in data_metadata[1]
                }
            )
            self.convs.append(conv)

            # BatchNorm untuk node transaction
            self.bns.append(nn.BatchNorm1d(hidden_channels))

        # Classifier linear bawaan (Sesuai Proposal Bab 3.4)
        self.classifier = nn.Linear(hidden_channels, out_channels)

    def forward(self, x_dict, edge_index_dict, num_nodes_dict, return_embeddings=False):
        # 1. Proyeksi fitur awal
        h_dict = {}

        # Node yang memiliki fitur (transaction, user, merchant)
        for node_type, x in x_dict.items():
            if node_type in self.lin_embeddings:
                h_dict[node_type] = self.lin_embeddings[node_type](x)
            elif node_type in self.param_embeddings:
                num_nodes = num_nodes_dict[node_type]
                h_dict[node_type] = self.param_embeddings[node_type].expand(
                    num_nodes, -1
                )

        # 2. Message Passing Layer
        for i in range(self.num_layers):
            h_dict = self.convs[i](h_dict, edge_index_dict)
            if "transaction" in h_dict and h_dict["transaction"].size(0) > 1:
                h_dict["transaction"] = self.bns[i](h_dict["transaction"])
            h_dict = {k: F.relu(v) for k, v in h_dict.items()}
            h_dict = {k: self.dropout(v) for k, v in h_dict.items()}

        tx_emb = h_dict["transaction"]

        if return_embeddings:
            return tx_emb

        logits = self.classifier(tx_emb)
        return logits.squeeze(-1)


def manual_weighted_bce(logits, targets, pos_weight):
    """
    Weighted Binary Cross Entropy Loss stabil dengan koefisien pos_weight teredam.
    """
    max_val = torch.clamp(-logits, min=0)
    loss = (1 - targets) * logits + (1 + (pos_weight - 1) * targets) * (
        max_val + torch.log(torch.exp(-max_val) + torch.exp(-logits - max_val))
    )
    return loss.mean()


def manual_unweighted_bce(logits, targets):
    """
    Standar Binary Cross Entropy Loss tanpa pembobotan untuk evaluasi objektif.
    """
    max_val = torch.clamp(-logits, min=0)
    loss = (1 - targets) * logits + (
        max_val + torch.log(torch.exp(-max_val) + torch.exp(-logits - max_val))
    )
    return loss.mean()


In [ ]:
%%writefile reproducibility.py
from __future__ import annotations

import hashlib
import json
import os
import platform
import random
import shutil
import subprocess
import sys
from contextlib import contextmanager
from pathlib import Path
from typing import Any, Iterator

import numpy as np
import torch


def seed_everything(
    seed: int, deterministic: bool = True, warn_only: bool = True
) -> None:
    """Seed all RNGs used by the EXP21E-v5 pipeline."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    if deterministic:
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        torch.use_deterministic_algorithms(True, warn_only=warn_only)


@contextmanager
def temporary_seed(seed: int) -> Iterator[None]:
    """Temporarily reset RNGs so validation/test sampling is repeatable."""
    python_state = random.getstate()
    numpy_state = np.random.get_state()
    torch_state = torch.random.get_rng_state()
    cuda_states = torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None
    seed_everything(seed)
    try:
        yield
    finally:
        random.setstate(python_state)
        np.random.set_state(numpy_state)
        torch.random.set_rng_state(torch_state)
        if cuda_states is not None:
            torch.cuda.set_rng_state_all(cuda_states)


def stable_hash(value: Any, length: int = 12) -> str:
    payload = json.dumps(value, sort_keys=True, ensure_ascii=False, default=str).encode(
        "utf-8"
    )
    return hashlib.sha256(payload).hexdigest()[:length]


def sha256_file(path: str | Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


def file_fingerprint(path: str | Path, with_sha256: bool = False) -> dict[str, Any]:
    file_path = Path(path).resolve()
    stat = file_path.stat()
    output: dict[str, Any] = {
        "path": str(file_path),
        "size_bytes": stat.st_size,
        "modified_time_ns": stat.st_mtime_ns,
    }
    if with_sha256:
        output["sha256"] = sha256_file(file_path)
    return output


def git_revision(workdir: str | Path) -> dict[str, Any]:
    root = Path(workdir).resolve()
    try:
        commit = subprocess.check_output(
            ["git", "rev-parse", "HEAD"], cwd=root, text=True, stderr=subprocess.DEVNULL
        ).strip()
        dirty = bool(
            subprocess.check_output(
                ["git", "status", "--porcelain"],
                cwd=root,
                text=True,
                stderr=subprocess.DEVNULL,
            ).strip()
        )
        return {"commit": commit, "dirty": dirty}
    except (OSError, subprocess.CalledProcessError):
        return {"commit": None, "dirty": None}


def environment_manifest() -> dict[str, Any]:
    try:
        import torch_geometric

        pyg_version = torch_geometric.__version__
    except Exception:
        pyg_version = None

    try:
        import pyg_lib

        pyg_lib_version = getattr(pyg_lib, "__version__", "installed")
    except Exception:
        pyg_lib_version = None

    return {
        "python": sys.version,
        "platform": platform.platform(),
        "numpy": np.__version__,
        "torch": torch.__version__,
        "torch_geometric": pyg_version,
        "pyg_lib": pyg_lib_version,
        "cuda_runtime": torch.version.cuda,
        "cuda_available": torch.cuda.is_available(),
        "cuda_device": torch.cuda.get_device_name(0)
        if torch.cuda.is_available()
        else None,
    }


def snapshot_source(source_dir: str | Path, destination: str | Path) -> None:
    """Copy the exact text/code assets used by a run into its result directory."""
    source = Path(source_dir).resolve()
    target = Path(destination).resolve()
    target.mkdir(parents=True, exist_ok=True)
    allowed_suffixes = {".py", ".yaml", ".yml", ".json", ".md", ".ipynb"}

    for item in source.iterdir():
        if item.name == "__pycache__":
            continue
        if item.is_file() and item.suffix.lower() in allowed_suffixes:
            shutil.copy2(item, target / item.name)


In [ ]:
%%writefile config.kaggle.yaml
experiment:
  id: "EXP21E"
  version: "v8"
  stage: "mechanism_audit_and_statistical_analysis"
  seed: 2108

data:
  transactions: "/kaggle/input/credit-card-transactions/credit_card_transactions-ibm_v2.csv"
  start_year: 2016

time_window:
  reference_date: "2018-01-01"
  train_days: 300
  val_days: 30
  gap_days: 8
  test_days: 60

features:
  mode: "clean"
  use_cyclical_time: true
  use_amount_log: true

artifacts:
  search_roots: ["/kaggle/input"]
  require_v5_runs: 15
  require_v6_runs: 18
  require_v7_runs: 5
  require_checkpoints: true

sampling:
  strategies: [uniform, topology, importance]
  smoothing_alpha: 0.1
  fanout: 10
  destination_groups_per_relation: 24
  empirical_draws: 2000
  audit_seed: 4108

statistics:
  bootstrap_replicates: 2000
  confidence_level: 0.95
  bootstrap_seed: 3108
  alert_budgets: [100, 200, 500, 1000]
  super_hub_quantile: 0.99

embedding_audit:
  enabled: true
  batch_size: 1024
  num_neighbors: [15, 10]
  probability_tolerance: 0.00001
  device: "auto"

output:
  result_dir: "/kaggle/working/result/exp21e/v8"
  snapshot_source: true



In [ ]:
%%writefile install_dependencies.py
from __future__ import annotations

import importlib.metadata
import importlib.util
import json
import subprocess
import sys
from pathlib import Path
from typing import Any


REPORT_PATH = Path("dependency_report.json")


def normalize_torch_version(version: str) -> str:
    """Return the three-component version used by the official PyG wheel index."""
    core = version.split("+", maxsplit=1)[0]
    parts = core.split(".")
    if len(parts) < 3 or not all(part.isdigit() for part in parts[:3]):
        raise ValueError(f"Unsupported torch version string: {version!r}")
    return ".".join(parts[:3])


def cuda_tag(cuda_version: str | None) -> str:
    """Convert torch's CUDA version (for example 12.8) to a PyG wheel tag."""
    if not cuda_version:
        return "cpu"
    parts = cuda_version.split(".")
    if len(parts) < 2 or not all(part.isdigit() for part in parts[:2]):
        raise ValueError(f"Unsupported CUDA version string: {cuda_version!r}")
    return f"cu{parts[0]}{parts[1]}"


def pyg_wheel_url(torch_version: str, cuda_version: str | None) -> str:
    return (
        "https://data.pyg.org/whl/"
        f"torch-{normalize_torch_version(torch_version)}+{cuda_tag(cuda_version)}.html"
    )


def module_available(name: str) -> bool:
    return importlib.util.find_spec(name) is not None


def package_version(name: str) -> str | None:
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None


def run_command(command: list[str], report: dict[str, Any]) -> None:
    printable = " ".join(command)
    print(f"[dependency-bootstrap] {printable}", flush=True)
    completed = subprocess.run(command, check=False, text=True)
    report["commands"].append({"command": command, "returncode": completed.returncode})
    if completed.returncode != 0:
        raise RuntimeError(f"Dependency installation failed ({completed.returncode}): {printable}")


def validate_runtime(report: dict[str, Any]) -> None:
    smoke_test = r"""
import torch
import torch_geometric
import pyg_lib
from torch_geometric.data import Data
from torch_geometric.loader import NeighborLoader

data = Data(
    x=torch.ones((4, 1), dtype=torch.float32),
    edge_index=torch.tensor([[0, 1, 2, 3, 0, 2], [1, 2, 3, 0, 2, 0]], dtype=torch.long),
)
data.sampling_weight = torch.ones(data.edge_index.size(1), dtype=torch.float32)
loader = NeighborLoader(
    data,
    num_neighbors=[2],
    input_nodes=torch.tensor([0, 1], dtype=torch.long),
    batch_size=2,
    shuffle=False,
    num_workers=0,
    weight_attr="sampling_weight",
)
next(iter(loader))
print({"torch": torch.__version__, "torch_geometric": torch_geometric.__version__})
"""
    run_command([sys.executable, "-c", smoke_test], report)


def write_report(report: dict[str, Any]) -> None:
    REPORT_PATH.write_text(json.dumps(report, indent=2), encoding="utf-8")


def main() -> None:
    import torch

    wheel_url = pyg_wheel_url(torch.__version__, torch.version.cuda)
    report: dict[str, Any] = {
        "status": "running",
        "python": sys.version,
        "torch": torch.__version__,
        "cuda_runtime": torch.version.cuda,
        "wheel_index": wheel_url,
        "commands": [],
        "before": {
            "torch_geometric": package_version("torch-geometric"),
            "pyg_lib": package_version("pyg-lib"),
            "torch_scatter": package_version("torch-scatter"),
            "torch_sparse": package_version("torch-sparse"),
        },
    }
    try:
        if not module_available("torch_geometric"):
            run_command(
                [
                    sys.executable,
                    "-m",
                    "pip",
                    "install",
                    "--disable-pip-version-check",
                    "--no-cache-dir",
                    "torch_geometric",
                ],
                report,
            )

        compiled_modules = {
            "pyg_lib": "pyg_lib",
            "torch_scatter": "torch_scatter",
            "torch_sparse": "torch_sparse",
        }
        missing = [package for package, module in compiled_modules.items() if not module_available(module)]
        if missing:
            run_command(
                [
                    sys.executable,
                    "-m",
                    "pip",
                    "install",
                    "--disable-pip-version-check",
                    "--no-cache-dir",
                    "--only-binary=:all:",
                    *missing,
                    "-f",
                    wheel_url,
                ],
                report,
            )

        validate_runtime(report)
        report["after"] = {
            "torch_geometric": package_version("torch-geometric"),
            "pyg_lib": package_version("pyg-lib"),
            "torch_scatter": package_version("torch-scatter"),
            "torch_sparse": package_version("torch-sparse"),
        }
        report["status"] = "passed"
    except Exception as exc:
        report["status"] = "failed"
        report["error"] = f"{type(exc).__name__}: {exc}"
        raise
    finally:
        write_report(report)

    print(json.dumps(report, indent=2), flush=True)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile artifact_io.py
from __future__ import annotations

import gzip
import hashlib
import json
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Iterable

import pandas as pd


@dataclass(frozen=True)
class RunArtifact:
    path: Path
    manifest: dict[str, Any]
    metrics: dict[str, Any]

    @property
    def version(self) -> str:
        return str(self.manifest.get("version", self.metrics.get("version", "")))

    @property
    def seed(self) -> int:
        return int(self.manifest.get("seed", self.metrics.get("seed")))

    @property
    def strategy(self) -> str:
        return str(
            self.manifest.get(
                "strategy",
                self.metrics.get("sampling_method", self.metrics.get("model_type", "unknown")),
            )
        )

    @property
    def window_id(self) -> str:
        return str(self.manifest.get("window_id", "primary_2018"))

    @property
    def run_id(self) -> str:
        return str(self.manifest.get("run_id", self.path.name))

    @property
    def checkpoint(self) -> Path:
        return self.path / "best_checkpoint.pt"

    @property
    def predictions(self) -> Path:
        return self.path / "predictions.csv.gz"


def read_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding="utf-8"))


def write_json(path: Path, value: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps(value, indent=2, ensure_ascii=False, sort_keys=False),
        encoding="utf-8",
    )


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def discover_runs(search_roots: Iterable[str | Path]) -> list[RunArtifact]:
    manifests: dict[Path, Path] = {}
    for raw_root in search_roots:
        root = Path(raw_root).expanduser().resolve()
        if not root.exists():
            continue
        for manifest_path in root.rglob("manifest.json"):
            run_dir = manifest_path.parent.resolve()
            if (run_dir / "metrics.json").exists() and (run_dir / "predictions.csv.gz").exists():
                manifests[run_dir] = manifest_path

    runs: list[RunArtifact] = []
    for run_dir, manifest_path in sorted(manifests.items(), key=lambda item: str(item[0])):
        manifest = read_json(manifest_path)
        metrics = read_json(run_dir / "metrics.json")
        if str(manifest.get("experiment_id", metrics.get("experiment_id", ""))) != "EXP21E":
            continue
        if str(manifest.get("status", "complete")) != "complete":
            continue
        runs.append(RunArtifact(run_dir, manifest, metrics))
    return runs


def validate_inventory(
    runs: list[RunArtifact],
    require_v5: int,
    require_v6: int,
    require_v7: int,
    require_checkpoints: bool,
) -> dict[str, Any]:
    expected = {"v5": require_v5, "v6": require_v6, "v7": require_v7}
    by_version = {version: [run for run in runs if run.version == version] for version in expected}
    errors: list[str] = []
    for version, count in expected.items():
        found = len(by_version[version])
        if found != count:
            errors.append(f"{version}: expected {count} complete runs, found {found}")
    for run in runs:
        if require_checkpoints and run.version == "v5" and not run.checkpoint.exists():
            errors.append(f"missing checkpoint: {run.run_id}")
        if not run.predictions.exists():
            errors.append(f"missing predictions: {run.run_id}")

    duplicate_keys: dict[tuple[str, str, str, int], list[str]] = {}
    for run in runs:
        key = (run.version, run.window_id, run.strategy, run.seed)
        duplicate_keys.setdefault(key, []).append(run.run_id)
    duplicates = {str(k): v for k, v in duplicate_keys.items() if len(v) > 1}
    if duplicates:
        errors.append(f"duplicate experiment cells: {duplicates}")

    report = {
        "status": "passed" if not errors else "failed",
        "counts": {version: len(items) for version, items in by_version.items()},
        "required": expected,
        "duplicates": duplicates,
        "errors": errors,
    }
    if errors:
        raise RuntimeError("Artifact inventory failed: " + "; ".join(errors))
    return report


def load_predictions(run: RunArtifact, split: str = "test") -> pd.DataFrame:
    frame = pd.read_csv(run.predictions, compression="gzip")
    required = {
        "split",
        "transaction_id",
        "timestamp",
        "label",
        "probability",
        "threshold",
        "prediction",
    }
    missing = required.difference(frame.columns)
    if missing:
        raise ValueError(f"{run.run_id} predictions missing columns: {sorted(missing)}")
    frame = frame.loc[frame["split"] == split].copy()
    frame["timestamp"] = pd.to_datetime(frame["timestamp"], errors="raise")
    frame["day"] = frame["timestamp"].dt.normalize()
    if frame["transaction_id"].duplicated().any():
        raise ValueError(f"Duplicate transaction_id in {run.run_id}/{split}")
    return frame.sort_values("transaction_id").reset_index(drop=True)


def validate_prediction_alignment(runs: list[RunArtifact], split: str = "test") -> dict[str, Any]:
    groups: dict[tuple[str, str], list[RunArtifact]] = {}
    for run in runs:
        if run.version in {"v5", "v7"}:
            groups.setdefault((run.window_id, split), []).append(run)
        elif run.version == "v6":
            groups.setdefault((run.window_id, split), []).append(run)

    report: dict[str, Any] = {"groups": {}, "passed": True}
    for key, group in sorted(groups.items()):
        reference_ids = None
        reference_labels = None
        reference_run = None
        entries: dict[str, Any] = {}
        for run in sorted(group, key=lambda item: (item.strategy, item.seed)):
            frame = load_predictions(run, split=split)
            ids = frame["transaction_id"].to_numpy()
            labels = frame["label"].to_numpy()
            if reference_ids is None:
                reference_ids, reference_labels, reference_run = ids, labels, run.run_id
            aligned = bool(
                len(ids) == len(reference_ids)
                and (ids == reference_ids).all()
                and (labels == reference_labels).all()
            )
            entries[run.run_id] = {"rows": len(frame), "aligned": aligned}
            report["passed"] = bool(report["passed"] and aligned)
        report["groups"]["/".join(key)] = {
            "reference_run": reference_run,
            "runs": entries,
        }
    if not report["passed"]:
        raise RuntimeError("Prediction alignment failed")
    return report


def artifact_manifest(runs: list[RunArtifact]) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    for run in sorted(runs, key=lambda item: (item.version, item.window_id, item.strategy, item.seed)):
        rows.append(
            {
                "version": run.version,
                "window_id": run.window_id,
                "strategy": run.strategy,
                "seed": run.seed,
                "run_id": run.run_id,
                "path": str(run.path),
                "metrics_sha256": sha256_file(run.path / "metrics.json"),
                "predictions_sha256": sha256_file(run.predictions),
                "checkpoint_sha256": sha256_file(run.checkpoint) if run.checkpoint.exists() else None,
            }
        )
    return rows


In [ ]:
%%writefile statistical_analysis.py
from __future__ import annotations

from collections import defaultdict
from typing import Any

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, f1_score, precision_recall_curve

from artifact_io import RunArtifact, load_predictions


def _metric_row(run: RunArtifact) -> dict[str, Any]:
    optimal = run.metrics["metrics_optimal_threshold"]
    independent = run.metrics["metrics_threshold_independent"]
    model = "tabular_mlp" if run.version == "v7" else run.strategy
    return {
        "version": run.version,
        "window_id": run.window_id,
        "model": model,
        "seed": run.seed,
        "run_id": run.run_id,
        "validation_auprc": float(run.metrics["hyperparameters"]["best_validation_auprc"]),
        "test_auprc": float(independent["auprc"]),
        "f1": float(optimal["f1_score"]),
        "precision": float(optimal["precision"]),
        "recall": float(optimal["recall"]),
        "alerts_per_captured_fraud": float(optimal["alerts_per_captured_fraud"]),
        "runtime_seconds": float(run.metrics["runtime_seconds"]["total"]),
    }


def main_results_table(runs: list[RunArtifact]) -> tuple[pd.DataFrame, pd.DataFrame]:
    per_run = pd.DataFrame([_metric_row(run) for run in runs if run.version in {"v5", "v6", "v7"}])
    primary = per_run.loc[per_run["version"].isin(["v5", "v7"])].copy()
    summary = (
        primary.groupby(["version", "window_id", "model"], sort=False)
        .agg(
            n_seeds=("seed", "nunique"),
            validation_auprc_mean=("validation_auprc", "mean"),
            validation_auprc_std=("validation_auprc", "std"),
            test_auprc_mean=("test_auprc", "mean"),
            test_auprc_std=("test_auprc", "std"),
            f1_mean=("f1", "mean"),
            f1_std=("f1", "std"),
            precision_mean=("precision", "mean"),
            recall_mean=("recall", "mean"),
            runtime_seconds_mean=("runtime_seconds", "mean"),
        )
        .reset_index()
    )
    return per_run, summary


def v6_robustness_tables(per_run: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    frame = per_run.loc[per_run["version"] == "v6"].copy()
    summary = (
        frame.groupby(["window_id", "model"], sort=True)
        .agg(
            n_seeds=("seed", "nunique"),
            validation_auprc_mean=("validation_auprc", "mean"),
            test_auprc_mean=("test_auprc", "mean"),
            test_auprc_std=("test_auprc", "std"),
            f1_mean=("f1", "mean"),
            f1_std=("f1", "std"),
        )
        .reset_index()
    )
    paired_rows: list[dict[str, Any]] = []
    for (window_id, paired_seed), cell in frame.groupby(["window_id", "seed"]):
        indexed = cell.set_index("model")
        if "uniform" not in indexed.index:
            raise RuntimeError(f"V6 uniform cell missing for {window_id}/seed{paired_seed}")
        for model in ("topology", "importance"):
            if model not in indexed.index:
                raise RuntimeError(f"V6 {model} cell missing for {window_id}/seed{paired_seed}")
            paired_rows.append(
                {
                    "window_id": window_id,
                    "seed": int(paired_seed),
                    "model": model,
                    "validation_auprc_delta_vs_uniform": float(
                        indexed.loc[model, "validation_auprc"]
                        - indexed.loc["uniform", "validation_auprc"]
                    ),
                    "test_auprc_delta_vs_uniform": float(
                        indexed.loc[model, "test_auprc"]
                        - indexed.loc["uniform", "test_auprc"]
                    ),
                    "f1_delta_vs_uniform": float(
                        indexed.loc[model, "f1"] - indexed.loc["uniform", "f1"]
                    ),
                }
            )
    return summary, pd.DataFrame(paired_rows)


def _daily_metric(frame: pd.DataFrame, metric: str) -> float:
    labels = frame["label"].to_numpy(dtype=np.int8)
    probabilities = frame["probability"].to_numpy(dtype=float)
    predictions = frame["prediction"].to_numpy(dtype=np.int8)
    if metric == "auprc":
        if np.unique(labels).size < 2:
            return float("nan")
        return float(average_precision_score(labels, probabilities))
    if metric == "f1":
        return float(f1_score(labels, predictions, zero_division=0))
    raise ValueError(metric)


def paired_daily_block_bootstrap(
    v5_runs: list[RunArtifact],
    v7_runs: list[RunArtifact],
    replicates: int,
    confidence_level: float,
    seed: int,
) -> dict[str, Any]:
    """Bootstrap paired *daily metric deltas*; the resampling unit is calendar day."""
    uniform = {run.seed: run for run in v5_runs if run.strategy == "uniform"}
    comparators: dict[str, dict[int, RunArtifact]] = {
        "topology": {run.seed: run for run in v5_runs if run.strategy == "topology"},
        "importance": {run.seed: run for run in v5_runs if run.strategy == "importance"},
        "tabular_mlp": {run.seed: run for run in v7_runs},
    }
    rng = np.random.default_rng(seed)
    alpha = 1.0 - confidence_level
    output: dict[str, Any] = {
        "unit_of_resampling": "calendar day, paired within seed and transaction-aligned model outputs",
        "estimand": "mean daily metric difference against V5 Uniform, averaged across paired seeds",
        "replicates": int(replicates),
        "confidence_level": float(confidence_level),
        "comparisons": {},
    }

    prediction_cache: dict[str, pd.DataFrame] = {}
    for run in list(uniform.values()) + [r for group in comparators.values() for r in group.values()]:
        prediction_cache[run.run_id] = load_predictions(run, split="test")

    for name, by_seed in comparators.items():
        paired_seeds = sorted(set(uniform).intersection(by_seed))
        if not paired_seeds:
            raise RuntimeError(f"No paired seeds for {name} versus uniform")
        comparison: dict[str, Any] = {"paired_seeds": paired_seeds, "metrics": {}}
        for metric in ("auprc", "f1"):
            seed_day_deltas: list[np.ndarray] = []
            per_seed: dict[str, Any] = {}
            for paired_seed in paired_seeds:
                base = prediction_cache[uniform[paired_seed].run_id]
                other = prediction_cache[by_seed[paired_seed].run_id]
                merged = base.merge(
                    other,
                    on="transaction_id",
                    suffixes=("_uniform", "_other"),
                    validate="one_to_one",
                )
                if not np.array_equal(merged["label_uniform"], merged["label_other"]):
                    raise RuntimeError(f"Label mismatch for {name}, seed {paired_seed}")
                merged["day"] = pd.to_datetime(merged["timestamp_uniform"]).dt.normalize()
                deltas: list[float] = []
                days: list[str] = []
                for day, day_frame in merged.groupby("day", sort=True):
                    base_frame = pd.DataFrame(
                        {
                            "label": day_frame["label_uniform"],
                            "probability": day_frame["probability_uniform"],
                            "prediction": day_frame["prediction_uniform"],
                        }
                    )
                    other_frame = pd.DataFrame(
                        {
                            "label": day_frame["label_other"],
                            "probability": day_frame["probability_other"],
                            "prediction": day_frame["prediction_other"],
                        }
                    )
                    base_value = _daily_metric(base_frame, metric)
                    other_value = _daily_metric(other_frame, metric)
                    if np.isfinite(base_value) and np.isfinite(other_value):
                        deltas.append(other_value - base_value)
                        days.append(pd.Timestamp(day).date().isoformat())
                array = np.asarray(deltas, dtype=float)
                if array.size == 0:
                    raise RuntimeError(f"No valid daily {metric} blocks for {name}, seed {paired_seed}")
                seed_day_deltas.append(array)
                per_seed[str(paired_seed)] = {
                    "days": len(days),
                    "mean_difference": float(array.mean()),
                }

            observed = float(np.mean([values.mean() for values in seed_day_deltas]))
            bootstrap = np.empty(replicates, dtype=float)
            for index in range(replicates):
                seed_means = [
                    float(values[rng.integers(0, len(values), size=len(values))].mean())
                    for values in seed_day_deltas
                ]
                bootstrap[index] = float(np.mean(seed_means))
            low, high = np.quantile(bootstrap, [alpha / 2.0, 1.0 - alpha / 2.0])
            comparison["metrics"][metric] = {
                "observed_mean_daily_difference": observed,
                "ci_low": float(low),
                "ci_high": float(high),
                "bootstrap_probability_positive": float((bootstrap > 0).mean()),
                "per_seed": per_seed,
            }
        output["comparisons"][name] = comparison
    return output


def add_causal_degrees(
    prediction: pd.DataFrame,
    history_frame: pd.DataFrame,
    test_frame: pd.DataFrame,
) -> pd.DataFrame:
    user_counts = history_frame["User"].value_counts().to_dict()
    merchant_counts = history_frame["Merchant Name"].value_counts().to_dict()
    degree_parts: list[pd.DataFrame] = []
    ordered = test_frame.sort_values(["datetime", "transaction_id"]).copy()
    ordered["day"] = ordered["datetime"].dt.normalize()
    for _, day_frame in ordered.groupby("day", sort=True):
        part = day_frame[["transaction_id", "User", "Merchant Name"]].copy()
        part["user_degree"] = part["User"].map(user_counts).fillna(0).astype(np.int64)
        part["merchant_degree"] = part["Merchant Name"].map(merchant_counts).fillna(0).astype(np.int64)
        degree_parts.append(part[["transaction_id", "user_degree", "merchant_degree"]])
        for value, count in day_frame["User"].value_counts().items():
            user_counts[value] = user_counts.get(value, 0) + int(count)
        for value, count in day_frame["Merchant Name"].value_counts().items():
            merchant_counts[value] = merchant_counts.get(value, 0) + int(count)
    degrees = pd.concat(degree_parts, ignore_index=True)
    return prediction.merge(degrees, on="transaction_id", how="left", validate="one_to_one")


def _quartiles(values: pd.Series) -> pd.Series:
    ranked = values.rank(method="first")
    return pd.qcut(ranked, 4, labels=["Q1", "Q2", "Q3", "Q4"])


def degree_stratified_analysis(
    runs: list[RunArtifact],
    history_frame: pd.DataFrame,
    test_frame: pd.DataFrame,
    super_hub_quantile: float,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    metric_rows: list[dict[str, Any]] = []
    error_rows: list[dict[str, Any]] = []
    for run in runs:
        frame = add_causal_degrees(load_predictions(run, "test"), history_frame, test_frame)
        for degree_name in ("user_degree", "merchant_degree"):
            frame["stratum"] = _quartiles(frame[degree_name])
            cutoff = float(frame[degree_name].quantile(super_hub_quantile))
            strata = [(str(label), group) for label, group in frame.groupby("stratum", observed=True)]
            strata.append(("super_hub", frame.loc[frame[degree_name] >= cutoff]))
            for stratum, group in strata:
                labels = group["label"].to_numpy(dtype=np.int8)
                probs = group["probability"].to_numpy(dtype=float)
                preds = group["prediction"].to_numpy(dtype=np.int8)
                auprc = float(average_precision_score(labels, probs)) if labels.sum() else float("nan")
                metric_rows.append(
                    {
                        "version": run.version,
                        "model": "tabular_mlp" if run.version == "v7" else run.strategy,
                        "seed": run.seed,
                        "degree_type": degree_name,
                        "stratum": stratum,
                        "rows": len(group),
                        "fraud": int(labels.sum()),
                        "auprc": auprc,
                        "f1": float(f1_score(labels, preds, zero_division=0)),
                    }
                )
                false_positive = (labels == 0) & (preds == 1)
                false_negative = (labels == 1) & (preds == 0)
                error_rows.append(
                    {
                        "version": run.version,
                        "model": "tabular_mlp" if run.version == "v7" else run.strategy,
                        "seed": run.seed,
                        "degree_type": degree_name,
                        "stratum": stratum,
                        "fp": int(false_positive.sum()),
                        "fn": int(false_negative.sum()),
                        "mean_absolute_margin": float(np.abs(probs - group["threshold"].to_numpy()).mean()),
                        "fp_mean_margin": float(np.abs(probs[false_positive] - group.loc[false_positive, "threshold"]).mean()) if false_positive.any() else None,
                        "fn_mean_margin": float(np.abs(probs[false_negative] - group.loc[false_negative, "threshold"]).mean()) if false_negative.any() else None,
                    }
                )
    return pd.DataFrame(metric_rows), pd.DataFrame(error_rows)


def prediction_case_analysis(v5_runs: list[RunArtifact]) -> tuple[pd.DataFrame, pd.DataFrame]:
    summaries: list[dict[str, Any]] = []
    cases: list[pd.DataFrame] = []
    for seed in sorted({run.seed for run in v5_runs}):
        selected = {run.strategy: run for run in v5_runs if run.seed == seed}
        if set(selected) != {"uniform", "topology", "importance"}:
            raise RuntimeError(f"Incomplete V5 strategy set for seed {seed}")
        merged = None
        for strategy in ("uniform", "topology", "importance"):
            frame = load_predictions(selected[strategy], "test")
            keep = frame[["transaction_id", "timestamp", "label", "probability", "prediction"]].rename(
                columns={"probability": f"probability_{strategy}", "prediction": f"prediction_{strategy}"}
            )
            merged = keep if merged is None else merged.merge(
                keep.drop(columns=["timestamp", "label"]), on="transaction_id", validate="one_to_one"
            )
        pred_cols = [f"prediction_{name}" for name in ("uniform", "topology", "importance")]
        probability_cols = [f"probability_{name}" for name in ("uniform", "topology", "importance")]
        merged["consistent_wrong"] = np.logical_and.reduce(
            [merged[column].to_numpy() != merged["label"].to_numpy() for column in pred_cols]
        )
        merged["prediction_switch"] = merged[pred_cols].nunique(axis=1) > 1
        merged["probability_range"] = merged[probability_cols].max(axis=1) - merged[probability_cols].min(axis=1)
        summaries.append(
            {
                "seed": seed,
                "rows": len(merged),
                "consistent_wrong": int(merged["consistent_wrong"].sum()),
                "prediction_switch": int(merged["prediction_switch"].sum()),
                "fraud_prediction_switch": int((merged["prediction_switch"] & (merged["label"] == 1)).sum()),
            }
        )
        selected_cases = merged.loc[merged["consistent_wrong"] | merged["prediction_switch"]].copy()
        selected_cases["seed"] = seed
        cases.append(selected_cases.sort_values("probability_range", ascending=False).head(250))
    return pd.DataFrame(summaries), pd.concat(cases, ignore_index=True)


def curve_tables(runs: list[RunArtifact], alert_budgets: list[int]) -> tuple[pd.DataFrame, pd.DataFrame]:
    pr_rows: list[dict[str, Any]] = []
    capture_rows: list[dict[str, Any]] = []
    recall_grid = np.linspace(0.0, 1.0, 201)
    grouped_precision: dict[str, list[np.ndarray]] = defaultdict(list)
    for run in runs:
        model = "tabular_mlp" if run.version == "v7" else run.strategy
        frame = load_predictions(run, "test")
        precision, recall, _ = precision_recall_curve(frame["label"], frame["probability"])
        order = np.argsort(recall)
        grouped_precision[model].append(np.interp(recall_grid, recall[order], precision[order]))
        ranked = frame.sort_values(["probability", "transaction_id"], ascending=[False, True])
        total_fraud = max(int(ranked["label"].sum()), 1)
        for budget in alert_budgets:
            top = ranked.head(min(int(budget), len(ranked)))
            capture_rows.append(
                {
                    "model": model,
                    "seed": run.seed,
                    "alert_budget": int(budget),
                    "captured_fraud": int(top["label"].sum()),
                    "fraud_capture_rate": float(top["label"].sum() / total_fraud),
                }
            )
    for model, curves in grouped_precision.items():
        matrix = np.vstack(curves)
        for index, recall_value in enumerate(recall_grid):
            pr_rows.append(
                {
                    "model": model,
                    "recall": float(recall_value),
                    "precision_mean": float(matrix[:, index].mean()),
                    "precision_std": float(matrix[:, index].std(ddof=1)) if len(matrix) > 1 else 0.0,
                }
            )
    return pd.DataFrame(pr_rows), pd.DataFrame(capture_rows)


In [ ]:
%%writefile mechanism_audit.py
from __future__ import annotations

from typing import Any

import numpy as np
import pandas as pd
from scipy.stats import spearmanr


def sampling_score(degree: np.ndarray, strategy: str, alpha: float) -> np.ndarray:
    degree = np.maximum(np.asarray(degree, dtype=float), 1.0)
    if strategy == "uniform":
        return np.ones_like(degree)
    if strategy == "topology":
        raw = 1.0 / np.log1p(degree)
    elif strategy == "importance":
        raw = np.log1p(degree)
    else:
        raise ValueError(strategy)
    minimum, maximum = float(raw.min()), float(raw.max())
    if np.isclose(minimum, maximum):
        return np.ones_like(raw)
    return alpha + (1.0 - alpha) * (raw - minimum) / (maximum - minimum)


def _weighted_without_replacement_counts(
    weights: np.ndarray,
    fanout: int,
    draws: int,
    rng: np.random.Generator,
    batch_draws: int = 128,
) -> np.ndarray:
    """Vectorized Plackett–Luce draws using Gumbel top-k sampling."""
    weights = np.asarray(weights, dtype=float)
    if len(weights) <= fanout:
        return np.full(len(weights), draws, dtype=np.int64)
    log_weights = np.log(np.maximum(weights, np.finfo(float).tiny))
    counts = np.zeros(len(weights), dtype=np.int64)
    completed = 0
    while completed < draws:
        current = min(batch_draws, draws - completed)
        uniforms = np.clip(rng.random((current, len(weights))), 1e-12, 1.0 - 1e-12)
        gumbel = -np.log(-np.log(uniforms))
        selected = np.argpartition(log_weights[None, :] + gumbel, -fanout, axis=1)[:, -fanout:]
        counts += np.bincount(selected.ravel(), minlength=len(weights))
        completed += current
    return counts


def _select_groups(
    groups: pd.core.groupby.generic.DataFrameGroupBy,
    fanout: int,
    count: int,
    max_candidates: int,
) -> list[Any]:
    sizes = groups.size()
    eligible = sizes.loc[(sizes > fanout) & (sizes <= max_candidates)].sort_values()
    if eligible.empty:
        raise RuntimeError(
            f"No destination group has candidate count in ({fanout}, {max_candidates}]"
        )
    indices = np.unique(np.linspace(0, len(eligible) - 1, min(count, len(eligible))).round().astype(int))
    return eligible.index[indices].tolist()


def mechanism_audit(
    train_frame: pd.DataFrame,
    strategies: list[str],
    alpha: float,
    fanout: int,
    groups_per_relation: int,
    empirical_draws: int,
    seed: int,
    max_candidates_per_group: int = 500,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, dict[str, Any]]:
    frame = train_frame[["transaction_id", "User", "Merchant Name"]].copy()
    user_degree = frame["User"].value_counts()
    merchant_degree = frame["Merchant Name"].value_counts()
    frame["user_degree"] = frame["User"].map(user_degree).astype(float)
    frame["merchant_degree"] = frame["Merchant Name"].map(merchant_degree).astype(float)
    relation_specs = {
        "transaction__rev_buys__user": ("User", "merchant_degree"),
        "transaction__rev_sells__merchant": ("Merchant Name", "user_degree"),
    }
    inclusion_rows: list[dict[str, Any]] = []
    degree_rows: list[dict[str, Any]] = []
    overlap_rows: list[dict[str, Any]] = []
    summary: dict[str, Any] = {
        "scope": (
            "Primary-window training graph; destination groups are deterministically sampled "
            f"from candidate counts ({fanout}, {max_candidates_per_group}]."
        ),
        "draws_per_group": int(empirical_draws),
        "fanout": int(fanout),
        "smoothing_alpha": float(alpha),
        "relations": {},
    }

    for relation_index, (relation, (destination_column, counterpart_degree_column)) in enumerate(relation_specs.items()):
        grouped = frame.groupby(destination_column, sort=False)
        selected_group_ids = _select_groups(
            grouped, fanout, groups_per_relation, max_candidates_per_group
        )
        relation_summary: dict[str, Any] = {"groups": len(selected_group_ids), "strategies": {}}
        for group_index, destination_id in enumerate(selected_group_ids):
            candidates = grouped.get_group(destination_id).sort_values("transaction_id").reset_index(drop=True)
            degrees = candidates[counterpart_degree_column].to_numpy(dtype=float)
            ids = candidates["transaction_id"].to_numpy(dtype=np.int64)
            selected_sets: dict[str, set[int]] = {}
            for strategy_index, strategy in enumerate(strategies):
                weights = sampling_score(degrees, strategy, alpha)
                rng = np.random.default_rng(seed + relation_index * 100_000 + group_index * 100 + strategy_index)
                counts = _weighted_without_replacement_counts(weights, fanout, empirical_draws, rng)
                empirical = counts / float(empirical_draws)
                intended_mass = weights / weights.sum()
                for tx_id, degree, weight, mass, probability in zip(
                    ids, degrees, weights, intended_mass, empirical
                ):
                    inclusion_rows.append(
                        {
                            "relation": relation,
                            "destination_group": str(destination_id),
                            "strategy": strategy,
                            "transaction_id": int(tx_id),
                            "counterpart_degree": float(degree),
                            "sampling_weight": float(weight),
                            "intended_weight_mass": float(mass),
                            "empirical_inclusion_probability": float(probability),
                        }
                    )
                degree_rows.append(
                    {
                        "relation": relation,
                        "destination_group": str(destination_id),
                        "strategy": strategy,
                        "candidate_count": len(candidates),
                        "pool_degree_mean": float(degrees.mean()),
                        "sampled_degree_mean": float(np.average(degrees, weights=counts)),
                        "sampled_degree_median": float(
                            np.median(np.repeat(degrees, counts))
                        ),
                    }
                )
                one_draw = _weighted_without_replacement_counts(
                    weights, fanout, 1, np.random.default_rng(seed + group_index)
                )
                selected_sets[strategy] = set(ids[one_draw > 0].tolist())
            uniform_set = selected_sets["uniform"]
            for strategy in strategies:
                if strategy == "uniform":
                    continue
                other = selected_sets[strategy]
                overlap_rows.append(
                    {
                        "relation": relation,
                        "destination_group": str(destination_id),
                        "strategy": strategy,
                        "intersection": len(uniform_set.intersection(other)),
                        "union": len(uniform_set.union(other)),
                        "jaccard": len(uniform_set.intersection(other)) / max(len(uniform_set.union(other)), 1),
                    }
                )

        inclusion_relation = pd.DataFrame(inclusion_rows)
        inclusion_relation = inclusion_relation.loc[inclusion_relation["relation"] == relation]
        for strategy in strategies:
            subset = inclusion_relation.loc[inclusion_relation["strategy"] == strategy]
            if subset["sampling_weight"].nunique() < 2:
                correlation = None
            else:
                value = spearmanr(
                    subset["sampling_weight"], subset["empirical_inclusion_probability"]
                ).statistic
                correlation = float(value) if np.isfinite(value) else None
            relation_summary["strategies"][strategy] = {
                "candidates": len(subset),
                "spearman_weight_vs_empirical_inclusion": correlation,
                "mean_empirical_inclusion": float(
                    subset["empirical_inclusion_probability"].mean()
                ),
            }
        summary["relations"][relation] = relation_summary

    return (
        pd.DataFrame(inclusion_rows),
        pd.DataFrame(degree_rows),
        pd.DataFrame(overlap_rows),
        summary,
    )


In [ ]:
%%writefile embedding_audit.py
from __future__ import annotations

import gc
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch

from artifact_io import RunArtifact, load_predictions
from causal_graph import CausalHistoryState, iter_causal_daily_days
from model import HeteroGraphSAGE
from reproducibility import temporary_seed
from samplers import get_neighbor_loader


def _load_checkpoint(path: Path) -> dict[str, Any]:
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")


def _device(name: str) -> torch.device:
    if name == "auto":
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    return torch.device(name)


def _initialize_model(
    checkpoint: dict[str, Any],
    train_graph,
    feature_columns: list[str],
    strategy: str,
    seed: int,
    device: torch.device,
) -> tuple[HeteroGraphSAGE, dict[str, Any]]:
    cfg = checkpoint["config"]
    model_cfg = cfg["model"]
    training_cfg = cfg["training"]
    sampling_cfg = cfg["sampling"]
    model = HeteroGraphSAGE(
        hidden_channels=int(model_cfg["hidden_channels"]),
        out_channels=int(model_cfg.get("out_channels", 1)),
        num_layers=int(model_cfg["num_layers"]),
        data_metadata=train_graph.metadata(),
        feature_dims={"transaction": len(feature_columns), "user": 2, "merchant": 2},
        aggr=model_cfg.get("aggr", "mean"),
        dropout=float(model_cfg.get("dropout", 0.2)),
    ).to(device)
    loader = get_neighbor_loader(
        train_graph,
        method=strategy,
        batch_size=2,
        shuffle=False,
        num_neighbors=list(training_cfg.get("num_neighbors", [15, 10])),
        seed=seed,
        alpha=float(sampling_cfg.get("smoothing_alpha", 0.1)),
        input_nodes=torch.arange(2),
    )
    with temporary_seed(seed), torch.no_grad():
        batch = next(iter(loader)).to(device)
        counts = {node_type: batch[node_type].num_nodes for node_type in batch.node_types}
        model(batch.x_dict, batch.edge_index_dict, counts)
    model.load_state_dict(checkpoint["model_state"], strict=True)
    model.eval()
    del loader, batch
    return model, cfg


def embedding_audit(
    v5_runs: list[RunArtifact],
    train_graph,
    processed_train: pd.DataFrame,
    processed_validation: pd.DataFrame,
    processed_gap: pd.DataFrame,
    processed_test: pd.DataFrame,
    feature_columns: list[str],
    entity_scalers: dict[str, Any],
    device_name: str,
    batch_size_override: int,
    probability_tolerance: float,
) -> pd.DataFrame:
    device = _device(device_name)
    rows: list[dict[str, Any]] = []
    history = (
        pd.concat([processed_train, processed_validation, processed_gap], ignore_index=True)
        .sort_values(["datetime", "transaction_id"])
        .reset_index(drop=True)
    )

    for run in sorted(v5_runs, key=lambda item: (item.strategy, item.seed)):
        checkpoint = _load_checkpoint(run.checkpoint)
        model, cfg = _initialize_model(
            checkpoint, train_graph, feature_columns, run.strategy, run.seed, device
        )
        training_cfg = cfg["training"]
        alpha = float(cfg["sampling"].get("smoothing_alpha", 0.1))
        neighbors = list(training_cfg.get("num_neighbors", [15, 10]))
        evaluation_seed = int(cfg["experiment"].get("evaluation_seed", run.seed + 1000)) + 10_000
        batch_size = int(batch_size_override or training_cfg.get("batch_size", 1024))
        state = CausalHistoryState(history.copy())
        plan = iter_causal_daily_days(
            state,
            processed_test,
            feature_columns,
            entity_scalers,
        )
        class_count = {0: 0, 1: 0}
        class_sum: dict[int, np.ndarray | None] = {0: None, 1: None}
        class_sum_norm2 = {0: 0.0, 1: 0.0}
        observed_ids: list[np.ndarray] = []
        observed_probabilities: list[np.ndarray] = []

        for day_index, causal_day in enumerate(plan):
            day_seed = evaluation_seed + day_index
            loader = get_neighbor_loader(
                causal_day.graph,
                method=run.strategy,
                batch_size=batch_size,
                shuffle=False,
                num_neighbors=neighbors,
                seed=day_seed,
                alpha=alpha,
                input_nodes=causal_day.root_nodes,
            )
            with temporary_seed(day_seed), torch.no_grad():
                for batch in loader:
                    root_count = batch["transaction"].batch_size
                    ids = batch["transaction"].transaction_id[:root_count].cpu().numpy()
                    batch = batch.to(device)
                    counts = {node_type: batch[node_type].num_nodes for node_type in batch.node_types}
                    embeddings = model(
                        batch.x_dict,
                        batch.edge_index_dict,
                        counts,
                        return_embeddings=True,
                    )[:root_count]
                    probabilities = torch.sigmoid(model.classifier(embeddings).squeeze(-1))
                    labels = batch["transaction"].y[:root_count].to(dtype=torch.int64)
                    embeddings_np = embeddings.detach().cpu().numpy().astype(np.float64)
                    labels_np = labels.cpu().numpy()
                    observed_ids.append(ids)
                    observed_probabilities.append(probabilities.cpu().numpy())
                    for class_id in (0, 1):
                        selected = embeddings_np[labels_np == class_id]
                        if not len(selected):
                            continue
                        class_count[class_id] += len(selected)
                        vector_sum = selected.sum(axis=0)
                        class_sum[class_id] = (
                            vector_sum
                            if class_sum[class_id] is None
                            else class_sum[class_id] + vector_sum
                        )
                        class_sum_norm2[class_id] += float(np.square(selected).sum())
            del loader

        if min(class_count.values()) == 0:
            raise RuntimeError(f"Both classes are required for embedding audit: {run.run_id}")
        mean0 = class_sum[0] / class_count[0]
        mean1 = class_sum[1] / class_count[1]
        centroid_distance = float(np.linalg.norm(mean1 - mean0))
        scatter0 = max(class_sum_norm2[0] - class_count[0] * float(mean0 @ mean0), 0.0)
        scatter1 = max(class_sum_norm2[1] - class_count[1] * float(mean1 @ mean1), 0.0)
        within_mean = scatter0 / class_count[0] + scatter1 / class_count[1]
        fisher_ratio = float(centroid_distance**2 / max(within_mean, np.finfo(float).eps))

        ids = np.concatenate(observed_ids)
        probabilities = np.concatenate(observed_probabilities)
        reproduced = pd.DataFrame({"transaction_id": ids, "reproduced_probability": probabilities})
        saved = load_predictions(run, "test")[["transaction_id", "probability"]]
        check = saved.merge(reproduced, on="transaction_id", validate="one_to_one")
        maximum_probability_error = float(
            np.max(np.abs(check["probability"] - check["reproduced_probability"]))
        )
        if maximum_probability_error > probability_tolerance:
            raise RuntimeError(
                f"Checkpoint reproduction failed for {run.run_id}: "
                f"max error {maximum_probability_error:.8g} > {probability_tolerance:.8g}"
            )
        rows.append(
            {
                "strategy": run.strategy,
                "seed": run.seed,
                "run_id": run.run_id,
                "embedding_dimension": len(mean0),
                "nonfraud_count": class_count[0],
                "fraud_count": class_count[1],
                "centroid_distance": centroid_distance,
                "within_class_scatter_mean": float(within_mean),
                "fisher_ratio": fisher_ratio,
                "max_probability_reproduction_error": maximum_probability_error,
            }
        )
        del model, checkpoint
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return pd.DataFrame(rows)


In [ ]:
%%writefile main.py
from __future__ import annotations

import argparse
import json
import shutil
import sys
import time
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

LOCAL_V5_SOURCE = Path(__file__).resolve().parent.parent / "exp21e_v5"
if LOCAL_V5_SOURCE.exists() and str(LOCAL_V5_SOURCE) not in sys.path:
    sys.path.insert(0, str(LOCAL_V5_SOURCE))

from artifact_io import (
    artifact_manifest,
    discover_runs,
    validate_inventory,
    validate_prediction_alignment,
    write_json,
)
from mechanism_audit import mechanism_audit
from statistical_analysis import (
    curve_tables,
    degree_stratified_analysis,
    main_results_table,
    paired_daily_block_bootstrap,
    prediction_case_analysis,
    v6_robustness_tables,
)


def load_config(path: Path) -> dict[str, Any]:
    return yaml.safe_load(path.read_text(encoding="utf-8"))


def resolve_dataset(configured: str) -> Path:
    path = Path(configured)
    if path.exists():
        return path.resolve()
    if Path("/kaggle/input").exists():
        matches = sorted(Path("/kaggle/input").rglob("credit_card_transactions-ibm_v2.csv"))
        if matches:
            return matches[0]
    raise FileNotFoundError(f"Dataset not found: {configured}")


def snapshot_sources(destination: Path, config_path: Path) -> None:
    destination.mkdir(parents=True, exist_ok=True)
    names = [
        "install_dependencies.py",
        "artifact_io.py",
        "statistical_analysis.py",
        "mechanism_audit.py",
        "embedding_audit.py",
        "main.py",
        "create_nb.py",
        "README.md",
        "kernel-metadata.json",
        config_path.name,
        "data_preprocessing.py",
        "graph_builder.py",
        "causal_graph.py",
        "samplers.py",
        "model.py",
        "reproducibility.py",
    ]
    source = Path(__file__).resolve().parent
    for name in names:
        candidate = source / name
        if not candidate.exists():
            candidate = LOCAL_V5_SOURCE / name
        if candidate.exists():
            shutil.copy2(candidate, destination / name)


def save_figures(
    output_dir: Path,
    pr_table: pd.DataFrame,
    capture_table: pd.DataFrame,
    inclusion: pd.DataFrame,
    degree_table: pd.DataFrame,
    error_table: pd.DataFrame,
) -> None:
    colors = {
        "uniform": "#4C78A8",
        "topology": "#F58518",
        "importance": "#54A24B",
        "tabular_mlp": "#B279A2",
    }
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for model, group in pr_table.groupby("model", sort=False):
        axes[0].plot(group["recall"], group["precision_mean"], label=model, color=colors.get(model))
        low = np.clip(group["precision_mean"] - group["precision_std"], 0, 1)
        high = np.clip(group["precision_mean"] + group["precision_std"], 0, 1)
        axes[0].fill_between(group["recall"], low, high, alpha=0.12, color=colors.get(model))
    axes[0].set(xlabel="Recall", ylabel="Precision", title="Mean precision–recall (±1 SD across seeds)")
    axes[0].legend()
    capture_summary = (
        capture_table.groupby(["model", "alert_budget"])["fraud_capture_rate"].mean().reset_index()
    )
    for model, group in capture_summary.groupby("model", sort=False):
        axes[1].plot(
            group["alert_budget"], group["fraud_capture_rate"], marker="o", label=model, color=colors.get(model)
        )
    axes[1].set(xlabel="Alert budget", ylabel="Fraud capture rate", title="Cumulative fraud capture")
    axes[1].legend()
    fig.tight_layout()
    fig.savefig(output_dir / "precision_recall_fraud_capture.png", dpi=220)
    plt.close(fig)

    weighted = inclusion.loc[inclusion["strategy"] != "uniform"].copy()
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for strategy, group in weighted.groupby("strategy"):
        sample = group.sample(min(5000, len(group)), random_state=2108)
        axes[0].scatter(
            sample["sampling_weight"], sample["empirical_inclusion_probability"],
            s=7, alpha=0.2, label=strategy, color=colors.get(strategy),
        )
    axes[0].set(
        xlabel="Intended sampling weight",
        ylabel="Empirical inclusion probability",
        title="Weight activation audit",
    )
    axes[0].legend()
    degree_summary = degree_table.groupby("strategy")[["pool_degree_mean", "sampled_degree_mean"]].mean()
    x = np.arange(len(degree_summary))
    width = 0.36
    axes[1].bar(x - width / 2, degree_summary["pool_degree_mean"], width, label="candidate pool")
    axes[1].bar(x + width / 2, degree_summary["sampled_degree_mean"], width, label="sampled")
    axes[1].set_xticks(x, degree_summary.index)
    axes[1].set(ylabel="Mean counterpart degree", title="Counterpart degree shift")
    axes[1].legend()
    fig.tight_layout()
    fig.savefig(output_dir / "sampler_mechanism.png", dpi=220)
    plt.close(fig)

    error_summary = (
        error_table.loc[error_table["stratum"] != "super_hub"]
        .groupby(["model", "degree_type", "stratum"])[["fp", "fn"]]
        .mean()
        .reset_index()
    )
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)
    for axis, degree_type in zip(axes, ("user_degree", "merchant_degree")):
        subset = error_summary.loc[error_summary["degree_type"] == degree_type]
        pivot = subset.pivot(index="stratum", columns="model", values="fn").reindex(["Q1", "Q2", "Q3", "Q4"])
        pivot.plot(kind="bar", ax=axis, color=[colors.get(column) for column in pivot.columns])
        axis.set(title=f"False negatives by {degree_type} quartile", xlabel="Stratum", ylabel="Mean FN")
        axis.tick_params(axis="x", rotation=0)
    fig.tight_layout()
    fig.savefig(output_dir / "degree_stratified_errors.png", dpi=220)
    plt.close(fig)


def build_report(
    summary: pd.DataFrame,
    v6_summary: pd.DataFrame,
    bootstrap: dict[str, Any],
    mechanism_summary: dict[str, Any],
    embedding_summary: pd.DataFrame | None,
) -> str:
    lines = [
        "# EXP21E-v8 Mechanism Audit and Statistical Analysis",
        "",
        "V8 is post-hoc only: no checkpoint, threshold, hyperparameter, or model weight was updated.",
        "",
        "## Main results",
        "",
        summary.to_markdown(index=False, floatfmt=".6f"),
        "",
        "## Rolling-window robustness",
        "",
        v6_summary.to_markdown(index=False, floatfmt=".6f"),
        "",
        "## Paired daily-block bootstrap",
        "",
        "The resampling unit is a calendar day. Effects are paired within seed against V5 Uniform.",
        "",
    ]
    for name, comparison in bootstrap["comparisons"].items():
        lines.append(f"### {name} versus Uniform")
        lines.append("")
        for metric, report in comparison["metrics"].items():
            lines.append(
                f"- {metric}: Δ={report['observed_mean_daily_difference']:.6f}, "
                f"{bootstrap['confidence_level']:.0%} CI "
                f"[{report['ci_low']:.6f}, {report['ci_high']:.6f}], "
                f"P(Δ>0)={report['bootstrap_probability_positive']:.3f}."
            )
        lines.append("")
    lines.extend(
        [
            "## Mechanism audit",
            "",
            f"Empirical draws per destination group: {mechanism_summary['draws_per_group']}.",
            "",
        ]
    )
    for relation, report in mechanism_summary["relations"].items():
        lines.append(f"### {relation}")
        lines.append("")
        for strategy, values in report["strategies"].items():
            lines.append(
                f"- {strategy}: Spearman(weight, inclusion)="
                f"{values['spearman_weight_vs_empirical_inclusion']}."
            )
        lines.append("")
    lines.extend(["## Embedding audit", ""])
    if embedding_summary is None:
        lines.append("Embedding audit was disabled by configuration.")
    else:
        aggregated = (
            embedding_summary.groupby("strategy")[["centroid_distance", "fisher_ratio"]]
            .agg(["mean", "std"])
            .reset_index()
        )
        lines.append(aggregated.to_markdown(index=False, floatfmt=".6f"))
        lines.append("")
        lines.append(
            "Checkpoint reproduction was enforced before representation statistics were accepted."
        )
    lines.extend(
        [
            "",
            "## Interpretation guardrail",
            "",
            "A mechanism claim is supported only when sampling weights alter inclusion/neighborhood "
            "statistics and the corresponding representation or error statistic changes consistently. "
            "Performance association alone is not interpreted as causation.",
            "",
        ]
    )
    return "\n".join(lines)


def main() -> None:
    parser = argparse.ArgumentParser(description="EXP21E-v8 post-hoc mechanism/statistical audit")
    parser.add_argument("--config", default="config.reference.yaml")
    args = parser.parse_args()
    config_path = Path(args.config).resolve()
    cfg = load_config(config_path)
    started = time.time()
    output_dir = Path(cfg["output"]["result_dir"]).resolve()
    output_dir.mkdir(parents=True, exist_ok=True)
    (output_dir / "figures").mkdir(exist_ok=True)
    (output_dir / "tables").mkdir(exist_ok=True)
    dependency_report = Path("dependency_report.json")
    if dependency_report.exists():
        shutil.copy2(dependency_report, output_dir / dependency_report.name)
    if cfg["output"].get("snapshot_source", True):
        snapshot_sources(output_dir / "code_snapshot", config_path)
    (output_dir / "resolved_config.yaml").write_text(
        yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True), encoding="utf-8"
    )

    artifact_cfg = cfg["artifacts"]
    runs = discover_runs(artifact_cfg["search_roots"])
    inventory = validate_inventory(
        runs,
        int(artifact_cfg["require_v5_runs"]),
        int(artifact_cfg["require_v6_runs"]),
        int(artifact_cfg["require_v7_runs"]),
        bool(artifact_cfg.get("require_checkpoints", True)),
    )
    alignment = validate_prediction_alignment(runs, split="test")
    write_json(output_dir / "artifact_inventory.json", inventory)
    write_json(output_dir / "prediction_alignment.json", alignment)
    pd.DataFrame(artifact_manifest(runs)).to_csv(output_dir / "artifact_manifest.csv", index=False)

    v5_runs = [run for run in runs if run.version == "v5"]
    v6_runs = [run for run in runs if run.version == "v6"]
    v7_runs = [run for run in runs if run.version == "v7"]
    per_run, main_table = main_results_table(runs)
    v6_summary, v6_paired = v6_robustness_tables(per_run)
    per_run.to_csv(output_dir / "tables" / "all_run_metrics.csv", index=False)
    main_table.to_csv(output_dir / "tables" / "main_results_table.csv", index=False)
    v6_summary.to_csv(output_dir / "tables" / "v6_robustness_summary.csv", index=False)
    v6_paired.to_csv(output_dir / "tables" / "v6_paired_differences.csv", index=False)

    stat_cfg = cfg["statistics"]
    bootstrap = paired_daily_block_bootstrap(
        v5_runs,
        v7_runs,
        int(stat_cfg["bootstrap_replicates"]),
        float(stat_cfg["confidence_level"]),
        int(stat_cfg["bootstrap_seed"]),
    )
    write_json(output_dir / "paired_daily_block_bootstrap.json", bootstrap)

    dataset_path = resolve_dataset(cfg["data"]["transactions"])
    from data_preprocessing import load_and_split_raw_data
    from reproducibility import environment_manifest, file_fingerprint, git_revision

    write_json(output_dir / "environment.json", environment_manifest())
    write_json(
        output_dir / "run_manifest.json",
        {
            "experiment": "EXP21E-v8",
            "stage": "mechanism_audit_and_statistical_analysis",
            "training_runs_started": 0,
            "dataset": file_fingerprint(dataset_path, with_sha256=False),
            "git": git_revision(Path(__file__).resolve().parent),
            "source_run_count": len(runs),
        },
    )

    time_cfg = cfg["time_window"]
    train, validation, test, gap, _ = load_and_split_raw_data(
        str(dataset_path),
        start_year=cfg["data"].get("start_year", 2016),
        reference_date=time_cfg["reference_date"],
        train_days=int(time_cfg["train_days"]),
        val_days=int(time_cfg["val_days"]),
        gap_days=int(time_cfg["gap_days"]),
        test_days=int(time_cfg["test_days"]),
    )
    raw_train, raw_validation, raw_test, raw_gap = (
        train.copy(), validation.copy(), test.copy(), gap.copy()
    )
    history = pd.concat([raw_train, raw_validation, raw_gap], ignore_index=True)
    degree_metrics, degree_errors = degree_stratified_analysis(
        v5_runs + v7_runs,
        history,
        raw_test,
        float(stat_cfg["super_hub_quantile"]),
    )
    degree_metrics.to_csv(output_dir / "tables" / "degree_stratified_metrics.csv", index=False)
    degree_errors.to_csv(output_dir / "tables" / "degree_stratified_errors.csv", index=False)

    case_summary, cases = prediction_case_analysis(v5_runs)
    case_summary.to_csv(output_dir / "tables" / "prediction_case_summary.csv", index=False)
    cases.to_csv(output_dir / "tables" / "prediction_switch_cases.csv.gz", index=False, compression="gzip")
    pr_table, capture_table = curve_tables(
        v5_runs + v7_runs, [int(value) for value in stat_cfg["alert_budgets"]]
    )
    pr_table.to_csv(output_dir / "tables" / "precision_recall_curve.csv", index=False)
    capture_table.to_csv(output_dir / "tables" / "fraud_capture_by_budget.csv", index=False)

    sampling_cfg = cfg["sampling"]
    inclusion, counterpart_degree, overlap, mechanism_summary = mechanism_audit(
        raw_train,
        list(sampling_cfg["strategies"]),
        float(sampling_cfg["smoothing_alpha"]),
        int(sampling_cfg["fanout"]),
        int(sampling_cfg["destination_groups_per_relation"]),
        int(sampling_cfg["empirical_draws"]),
        int(sampling_cfg["audit_seed"]),
        int(sampling_cfg.get("max_candidates_per_group", 500)),
    )
    inclusion.to_csv(output_dir / "tables" / "weight_vs_empirical_inclusion.csv.gz", index=False, compression="gzip")
    counterpart_degree.to_csv(output_dir / "tables" / "sampled_counterpart_degree.csv", index=False)
    overlap.to_csv(output_dir / "tables" / "sampled_edge_overlap.csv", index=False)
    write_json(output_dir / "sampler_mechanism_report.json", mechanism_summary)

    embedding_table = None
    embedding_cfg = cfg["embedding_audit"]
    if bool(embedding_cfg.get("enabled", True)):
        from data_preprocessing import apply_strict_anti_leakage_preprocessing
        from embedding_audit import embedding_audit
        from graph_builder import build_hetero_graph

        processed = apply_strict_anti_leakage_preprocessing(
            train,
            validation,
            test,
            gap,
            feature_mode=cfg["features"]["mode"],
            use_cyclical_time=bool(cfg["features"]["use_cyclical_time"]),
            use_amount_log=bool(cfg["features"]["use_amount_log"]),
        )
        train, validation, test, gap, feature_columns, _ = processed
        train_graph, entity_scalers = build_hetero_graph(train, feature_columns, scalers_dict=None)
        embedding_table = embedding_audit(
            v5_runs,
            train_graph,
            train,
            validation,
            gap,
            test,
            feature_columns,
            entity_scalers,
            str(embedding_cfg.get("device", "auto")),
            int(embedding_cfg.get("batch_size", 1024)),
            float(embedding_cfg.get("probability_tolerance", 1e-5)),
        )
        embedding_table.to_csv(output_dir / "tables" / "embedding_separation.csv", index=False)

    save_figures(
        output_dir / "figures",
        pr_table,
        capture_table,
        inclusion,
        counterpart_degree,
        degree_errors,
    )
    report = build_report(main_table, v6_summary, bootstrap, mechanism_summary, embedding_table)
    (output_dir / "EXP21E_V8_REPORT.md").write_text(report, encoding="utf-8")
    acceptance = {
        "status": "passed",
        "training_runs_started": 0,
        "inventory_passed": inventory["status"] == "passed",
        "prediction_alignment_passed": bool(alignment["passed"]),
        "figures": sorted(path.name for path in (output_dir / "figures").glob("*.png")),
        "machine_readable_statistical_report": True,
        "embedding_checkpoint_reproduction_passed": (
            None
            if embedding_table is None
            else bool(
                (embedding_table["max_probability_reproduction_error"] <= float(embedding_cfg["probability_tolerance"])).all()
            )
        ),
        "runtime_seconds": time.time() - started,
    }
    statistical_report = {
        "experiment": "EXP21E-v8",
        "status": "complete",
        "training_runs_started": 0,
        "inventory": inventory,
        "prediction_alignment_passed": bool(alignment["passed"]),
        "primary_results": main_table.to_dict(orient="records"),
        "v6_robustness": {
            "summary": v6_summary.to_dict(orient="records"),
            "paired_differences": v6_paired.to_dict(orient="records"),
        },
        "paired_daily_block_bootstrap": bootstrap,
        "sampler_mechanism": mechanism_summary,
        "embedding_summary": (
            None if embedding_table is None else embedding_table.to_dict(orient="records")
        ),
        "confidence_interval_unit": "calendar-day block paired within seed",
    }
    write_json(output_dir / "v8_statistical_report.json", statistical_report)
    write_json(output_dir / "acceptance_report.json", acceptance)
    print(json.dumps(acceptance, indent=2))


if __name__ == "__main__":
    main()


In [ ]:
import subprocess
import sys
subprocess.run([sys.executable, 'install_dependencies.py'], check=True)


In [ ]:
import subprocess
import sys
subprocess.run([sys.executable, 'main.py', '--config', 'config.kaggle.yaml'], check=True)
